# Figures S16–S22 — individual BO runs

Drawn by the original generic notebook `paperAnalysis/BO/bo_figure_publi.ipynb`, whose code cells are vendored verbatim in `bo_publi_cells.py` (no patch; the initial design — projected LHS — is labelled 'Initialization'). `toofast_analysis.bo_publi` executes them once per run on raw/ (configuration in `bo_publi.RUNS`: PCA bases from `data/inputs/`, targets from raw/, S21 stopped at evaluation 15).

| SI | run | panels |
|---|---|---|
| S16 | S04-M03 (2 pumps) | gp_mean_2pumps, cost_evolution, best_spectrum |
| S17 | S04-M02 (3 pumps) | gp_mean_ternary_3pumps, cost_evolution, best_spectrum |
| S18 | S19-M04 (3 pumps, PCA-norm L2) | gp_mean_ternary_3pumps, cost_evolution, best_spectrum_normalized |
| S19 | S22-M03 (3 pumps, PCA cosine) | gp_mean_ternary_3pumps, cost_evolution, best_spectrum |
| S20 | S08-M02 (4 pumps) | gp_mean_corner, cost_evolution, best_spectrum |
| S21 | S25-M03 (5 pumps, PCA cosine) | gp_mean_corner, cost_evolution, best_spectrum |
| S22 | S28-M02 (5 pumps, PCA cosine) | gp_mean_corner, cost_evolution, best_spectrum |

Evaluation numbers on the figures are 0-based (as in the SI). ≈ 10 min.

## Analysis code
All the code of this figure is in this notebook.

### `paths`

In [ ]:
import sys
import types

from IPython.core.magic import register_cell_magic

MODULE_SOURCE = {}


@register_cell_magic
def module(line, cell):
    """`%%module a.b`: run this cell as the Python module a.b, so that the original code keeps its own namespaces."""
    name = line.strip()
    parts = name.split(".")
    for i in range(1, len(parts)):                              # parent packages
        pkg = ".".join(parts[:i])
        if pkg not in sys.modules:
            sys.modules[pkg] = types.ModuleType(pkg)
            sys.modules[pkg].__path__ = []
            if i > 1:
                setattr(sys.modules[".".join(parts[:i - 1])], parts[i - 1], sys.modules[pkg])
    mod = types.ModuleType(name)
    mod.__package__ = name.rpartition(".")[0]
    sys.modules[name] = mod
    setattr(sys.modules[mod.__package__], parts[-1], mod)
    MODULE_SOURCE[name] = cell
    exec(compile(cell, f"<{name}>", "exec"), mod.__dict__)


In [ ]:
%%module toofast_analysis.paths
"""Repository paths."""
import os
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())   # notebook: no __file__
# raw/ can live elsewhere (e.g. an extracted Zenodo archive): set TOOFAST_RAW
RAW = Path(os.environ.get("TOOFAST_RAW", ROOT / "raw"))
PROCESSED = ROOT / "data" / "processed"
INPUTS = ROOT / "data" / "inputs"      # frozen artefacts used by the platform during the runs (versioned, read-only)
RESOURCE_DIR = INPUTS
FIGURES = ROOT / "figures"


In [ ]:
from toofast_analysis.paths import *

### `bo_publi_cells`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.bo_publi_cells
"""[vendored] code cells of bo_figure_publi.ipynb, verbatim from gen_bo_figure_publi.py (md5 e03759c501480a3adcaefecf5328eb26),
generated by tools/vendor_bo_publi.py — do not edit by hand. Executed by toofast_analysis.bo_publi.
No patch (legend label of the initial design: "Initialization (n)")."""
SOURCE_MD5 = 'e03759c501480a3adcaefecf5328eb26'
CELLS = [
    "import json\nimport pickle\nimport re\nimport warnings\nfrom pathlib import Path\n\nimport numpy as np\nimport matplotlib as mpl\nimport matplotlib.patheffects as pe\nimport matplotlib.pyplot as plt\nfrom matplotlib.colors import Normalize\nfrom mpl_toolkits.mplot3d import Axes3D  # noqa: F401\nfrom mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection\n\nwarnings.filterwarnings('ignore')\n\ntry:\n    import h5py\n    HAS_H5 = True\nexcept ImportError:\n    HAS_H5 = False\n    print('h5py missing -> H5 cross-check disabled (pip install h5py)')\n\ntry:\n    import plotly.graph_objects as go\n    import plotly.io as pio\n    HAS_PLOTLY = True\nexcept ImportError:\n    HAS_PLOTLY = False\n    print('plotly missing -> interactive HTML disabled (pip install plotly)')\n\n# ---- publication style -------------------------------------------------\nPUB_STYLE = {\n    'font.family': 'sans-serif',\n    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],\n    'font.size': 17,\n    'axes.labelsize': 20,\n    'axes.titlesize': 20,\n    'xtick.labelsize': 16,\n    'ytick.labelsize': 16,\n    'legend.fontsize': 15,\n    'figure.titlesize': 22,\n    'axes.linewidth': 1.6,\n    'xtick.major.width': 1.6, 'ytick.major.width': 1.6,\n    'xtick.major.size': 7, 'ytick.major.size': 7,\n    'xtick.minor.width': 1.0, 'ytick.minor.width': 1.0,\n    'lines.linewidth': 2.4,\n    'figure.dpi': 110,\n    'savefig.dpi': 400,\n    'savefig.bbox': 'tight',\n    'savefig.facecolor': 'white',\n    'mathtext.default': 'regular',\n    'axes.unicode_minus': False,\n}\nmpl.rcParams.update(PUB_STYLE)\nnp.set_printoptions(precision=4, suppress=True)\nFLOW_UNIT = '$\\\\mu$L s$^{-1}$'   # pump_flows.csv / _config.json are in uL/s\nFIG_SIZE = 9.6            # all single-panel publication figures are square\nprint('imports OK')",
    '# ---- the run folder to analyse ----------------------------------------\nEXPERIMENT_PATH = Path(r\'C:\\Users\\user\\paperAnalysis\\BO\\S28-M01\')\n\n# ---- which GP snapshot ------------------------------------------------\nGP_ITER = None          # None = last gaussian_processes/gp_model_iter_NNN.pkl\nSTOP_AT_EVAL = None     # cut the run at this evaluation number (as printed on\n                        # the figures, so POINT_LABEL_BASE applies): the GP\n                        # snapshot that includes it is loaded and every\n                        # figure - points, best, spectra, legend counts -\n                        # stops there. Overrides GP_ITER. E.g. S25-M03\'s\n                        # best is #15 -> STOP_AT_EVAL = 15.\n                        # int  = that iteration (e.g. 8)\n\n# ---- varying-pump detection -------------------------------------------\nVARYING_FROM = \'config\'  # \'config\' (fixed_pumps) | \'std\' (spread of pump_flows)\nVARYING_STD_EPS = 0.05   # used when VARYING_FROM == \'std\'\n\n# ---- what is drawn ----------------------------------------------------\nSHOW_GHOSTS = True       # virtual observations (penalty points the optimiser\n                         # adds to the GP training set at known-bad edges)\nSHOW_POINT_NUMBERS = True   # label every measurement with its evaluation order\nPOINT_LABEL_BASE = 0        # 0 -> points numbered 0..N-1, 1 -> 1..N\nCI_FACTOR = 2.0          # width of the confidence band on the 1-D figure\nCMAP_MU = \'viridis_r\'    # low cost = bright  (cost is minimised)\nCMAP_SD = \'magma\'\nCOST_LABEL = \'GP predicted cost\'\nCOLOR_SCALE = \'linear\'   # \'linear\' | \'log\' - log10(cost) colour mapping, useful\n                         # when costs span orders of magnitude (mostly the 4D+\n                         # tetrahedron clouds, where a linear scale washes out\n                         # the low-cost basin against a few very high outliers)\nTETRA_ALPHA_MODE = \'deviation\'  # \'deviation\' = fade out near-mean mu (both the\n                         # low-cost basin AND high-cost regions stand out\n                         # against a near-invisible "typical" bulk) | \'low\' =\n                         # old behaviour, opacity ramps with distance below max\n\n# ---- tetrahedron depth cues -------------------------------------------\n# A wireframe tetrahedron with 6 identical edges is a Necker cube: the eye\n# reads it as a flat triangle crossed by lines. These flags are what make it\n# read as a solid.\nTETRA_HIDDEN_LINES = True   # back edges thin/dashed/grey, front edges solid\n                         # black (exact back-face test, not a depth heuristic)\nTETRA_BACKFACE_ALPHA = 0.055   # light fill on the back faces -> enclosed\n                         # volume. 0 disables.\nTETRA_SHADOWS = True     # drop lines from every measurement to the base face\n                         # plus a grey ground shadow. This is what makes a 3-D\n                         # marker *localisable* - without it a point inside a\n                         # volume has no determinate position on the page.\nTETRA_TICK_EDGES = \'front\'  # \'all\' = 3 labels on each of the 6 edges (18\n                         # numbers, the old clutter) | \'front\' = silhouette\n                         # edges only | \'none\' = rely on the caption\nTETRA_LABEL_POINTS = \'key\'  # \'all\' = number every evaluation (they overlap in\n                         # a converged cluster) | \'key\' = first / best / last\n                         # only | \'none\'\nTETRA_PERSPECTIVE = None  # None = orthographic (exact fraction reading) |\n                         # float = matplotlib focal_length, e.g. 0.6, which\n                         # restores converging edges at a slight metric cost\nTETRA_CLOUD_Q = None     # None = keep the whole cloud (old behaviour) | float\n                         # = keep only that best quantile of mu, drawn denser\n                         # and more opaque. 0.1 turns the flat haze into one\n                         # readable low-cost blob.\n\n# ---- flat spectra (no plasmon band) -----------------------------------\n# production flags a measurement as flat when std(absorbance) < threshold,\n# then remaps its cost through the flat strategy. Display only here: the GP\n# surface stays exactly the one the run fitted.\nFLAT_STD_THRESHOLD = None   # None = _config.json \'threshold\'\nFLAT_STRATEGY = None        # None = _config.json \'flat_strategy_override\'\nFLAT_PENALTY_FACTOR = None  # None = _config.json \'flat_penalty_factor_override\'\nFLAT_NEAREST_MARGIN = None  # None = _config.json \'flat_nearest_margin_override\'\nFLAT_NEAREST_CAP = None     # None = _config.json \'flat_nearest_cap_override\'\nFLAT_NEAREST_RADIUS = None  # None = 1.5 x median nearest-neighbour distance\n\n# ---- best-spectrum figure ---------------------------------------------\nSPECTRUM_XLIM = (330.0, 950.0)   # None = full detector range\nTARGET_IS_SPECTRUM = None        # None = auto from target_style\n\n# ---- PCA-style target (target_style == \'pca\') --------------------------\n# The PCA decomposition and the target spectrum live OUTSIDE the run folder\n# and are shared across a family of runs, so they can\'t be auto-discovered -\n# set both paths explicitly when analysing a pca-target run (the decomposition\n# file path is logged in <name>_parameters.txt as "PCA decomposition file";\n# it DIFFERS per run family - e.g. S19 used pca_model_norm6.npz, S22 used\n# pca_model_5.npz - and the two even use different .npz key schemas, so don\'t\n# assume one file works for another family without checking parameters.txt).\n# Metric/component count default to the run\'s own _config.json\n# (pca_cost_metric, pca_n_components) - only override if that\'s absent or\n# a cross-check against costs.csv (see PCA TARGET summary block) disagrees.\n# Confirmed exactly (corr = 1.0000, ratio 1.0000+/-0.0000) for S22-M03\n# (pca_model_5.npz, normalize=\'none\', cosine, 4/5 components) and closely\n# (corr = 1.0000, ratio 0.9996+/-0.0007) for S19-M04 (pca_model_norm6.npz,\n# normalize=\'max\', unweighted L2, all 6 components - that run\'s config had no\n# pca_cost_metric/pca_n_components keys at all, hence the defaults). Both use\n# the LAST raw measurement row (not an averaged spectrum) for the candidate\n# and the target. The Huber-delta robustness parameter (grouped with the\n# flat-strategy overrides, not the pca_cost_metric/pca_n_components pair) is\n# NOT part of the primary cost in either case - flat-spectrum fallback only.\nPCA_MODEL_PATH = None     # e.g. r\'C:\\Users\\user\\2FAST_ML\\PCA\\pca_model_5.npz\'\nPCA_TARGET_PATH = None    # e.g. r\'C:\\...\\S22-M01\\measurement_spectra_iter_1.csv\'\nPCA_N_COMPONENTS = None   # None = CFG[\'pca_n_components\'], else every component\nPCA_COST_METRIC = None    # None = CFG[\'pca_cost_metric\'] (\'l2\'/\'cosine\'), else \'l2\'\n\n# ---- resolutions ------------------------------------------------------\nRES_TERNARY = 130        # subdivisions of the triangle edge (3 pumps)\nRES_SURF3D = 140         # Plotly surface grid (3 pumps)\nRES_VOLUME = 34          # Plotly volume grid per axis (4 pumps)\nRES_VOLUME_5D = 24       # Plotly volume grid per axis (5 pumps, per slice)\nN_SLICES_5D = 9          # slider positions (5 pumps)\n\n# ---- interactive 3-D rendering (4 / 5 pumps) --------------------------\nINTERACTIVE_3D_MODE = \'cloud\'  # \'cloud\' = dense point cloud restricted to the\n                         # informative region (default - far more readable than\n                         # nested translucent isosurfaces, which pile up into a\n                         # uniform haze) | \'volume\' = the go.Volume isosurface\n                         # rendering, kept for comparison\nCLOUD_N_4D = 90000       # points sampled on the simplex before filtering\nCLOUD_N_5D = 35000       # ... per slider slice (5 pumps)\n# Nested shells (quantile, opacity, marker size), innermost first. Each shell\n# is the BAND between the previous quantile and its own, so a single flat blob\n# becomes a bright compact core fading into a translucent halo - that gradient\n# is what makes the basin\'s shape and depth legible. The largest quantile is\n# the overall cut: everything beyond it is dropped as uninformative fog.\nCLOUD_SHELLS = ((0.03, 0.85, 4.2),\n                (0.12, 0.28, 2.8),\n                (0.32, 0.07, 2.0))\nCLOUD_MAX_PTS = 4500     # cap per shell. The outer shells only provide\n                         # context: left uncapped they stack thousands of\n                         # translucent points along every view ray and the\n                         # accumulated alpha turns the halo back into a solid\n                         # blob (and bloats the HTML). The core is naturally\n                         # small, so it is never capped.\nVOLUME_FOCUS_Q = 0.55    # isosurfaces cover the best 55 % of the cost range\n                         # (1.0 = render the whole range, muddier)\nSLICE_DIM_5D = None      # None = least influential axis; else index in varying pumps\nN_CLOUD = 60000          # points in the tetrahedron cloud (4 pumps)\nN_PROJ = 300000          # samples used for the min-projections (5 pumps)\nPROJ_BINS = 42           # binning of the min-projections\nTETRA_KEEP = None        # None = auto (drop least influential); else 4 indices\nTERNARY_KEEP = None      # None = auto (3 most influential); else 3 indices\n\n# ---- isopleth strip (the 2-D reading of a 4-D GP) ---------------------\n# One reagent is held at a series of fixed fractions; the other three are\n# renormalised onto a ternary triangle. With 4 varying pumps this is an\n# *exact section* of the GP (no projection, no binning) - the quaternary\n# isopleth of the materials literature. Any reagent beyond the 4 is\n# min-projected, exactly as in `fig_proj_ternary`.\nISOPLETH_HOLD = None     # None = most influential reagent; else index in the\n                         # varying pumps\nISOPLETH_LEVELS = (0.0, 0.1, 0.2, 0.3, 0.4)   # held fractions, one panel each\nISOPLETH_RES = 90        # ternary lattice subdivisions per panel\nISOPLETH_TOL = 0.06      # a measurement is drawn on the panel whose held\n                         # fraction it is within +/- this of\nISOPLETH_PANEL = 3.15    # inches per panel\n\n# ---- output -----------------------------------------------------------\nOUTPUT_SUBDIR = \'figures_publi\'\nSAVE_PNG = True\nSAVE_HTML = True\nSAVE_FORMATS = []        # extra formats per static figure: any of\n                         # \'tiff\' (raster, LZW-compressed) | \'pdf\' | \'svg\' | \'eps\'\n                         # (vector - the mu/sd point clouds rasterise inside\n                         # them since they\'re plotted as scatter, but axes,\n                         # text and contours stay vector)\nTIFF_DPI = 400           # most journals want >= 300-600 dpi for TIFF\nRANDOM_SEED = 0\n\nRNG = np.random.default_rng(RANDOM_SEED)\nOUTPUT_DIR = EXPERIMENT_PATH / OUTPUT_SUBDIR\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\nprint(\'run    :\', EXPERIMENT_PATH)\nprint(\'output :\', OUTPUT_DIR)',
    'PRETTY = {\n    \'AgNO3\': \'AgNO$_3$\', \'NaBH4\': \'NaBH$_4$\', \'PVP\': \'PVP\',\n    \'Acetic Acid\': \'Acetic acid\', \'Ascorbic Acid\': \'Ascorbic acid\',\n    \'H2O\': \'H$_2$O\', \'HNO3\': \'HNO$_3$\',\n}\n\n\ndef pretty(name):\n    return PRETTY.get(str(name).strip(), str(name).strip())\n\n\ndef _decode(x):\n    if isinstance(x, bytes):\n        return x.decode(\'utf-8\', \'replace\')\n    return x\n\n\ndef load_config(path):\n    p = Path(path) / \'_config.json\'\n    if not p.exists():\n        raise FileNotFoundError(f\'no _config.json in {path}\')\n    with open(p, encoding=\'utf-8\') as f:\n        return json.load(f)\n\n\ndef load_h5_meta(path):\n    \'\'\'Read the experiment metadata of the last sample H5 (cross-check source).\'\'\'\n    if not HAS_H5:\n        return {}\n    files = sorted((Path(path) / \'samples\').glob(\'*.h5\')) if (Path(path) / \'samples\').is_dir() else []\n    if not files:\n        return {}\n    # prefer the highest iter_N, else the last init_N\n    def _key(f):\n        m = re.search(r\'_(init|iter)_(\\d+)\\.h5$\', f.name)\n        if not m:\n            return (0, 0)\n        return (1 if m.group(1) == \'iter\' else 0, int(m.group(2)))\n    src = max(files, key=_key)\n    out = {\'_file\': src.name}\n    names = {\'residence_time_s\': \'residence_time\', \'total_flow_uL_s\': \'total_flow\',\n             \'integration_time_ms\': \'spectro_integration_time_ms\', \'n_averages\': \'spectro_number_averages\',\n             \'sampling_interval_s\': \'spectro_sampling_rate_s\', \'smoothing_px\': \'spectro_smoothing_px\',\n             \'target_wavelength_nm\': \'target_wavelength\', \'target_extinction\': \'target_absorbance\'}\n    with h5py.File(src, \'r\') as f:\n        for grp in (\'conditions\', \'instrument\', \'run_settings\'):\n            for k, v in f[grp].attrs.items():\n                out[names.get(k, k)] = _decode(v)\n        g = f[\'conditions\']\n        for k, key in ((\'varying_pumps\', \'varying_pumps\'), (\'flow_set\', \'flows_set\'), (\'reagents\', \'reagents\')):\n            if k in g:\n                out[key] = np.array([_decode(v) for v in g[k][()]]) \\\n                    if g[k].dtype.kind == \'S\' else np.array(g[k][()])\n        if \'varying_pumps\' in g:\n            out[\'fixed_pumps\'] = ~np.array(g[\'varying_pumps\'][()])\n        for k, v in f[\'sample\'].attrs.items():\n            out[\'meta_\' + k] = _decode(v)\n    return out\n\n\ndef load_params_txt(path):\n    \'\'\'Parse <name>_parameters.txt (free-text log written at the end of the run).\'\'\'\n    path = Path(path)\n    files = list(path.glob(\'*_parameters.txt\'))\n    if not files:\n        return {}\n    txt = files[0].read_text(encoding=\'utf-8\', errors=\'replace\')\n    out = {\'_file\': files[0].name, \'_raw\': txt}\n    pats = {\n        \'result\': r\'Results:\\s*(.+)\',\n        \'optimal_flows\': r\'Optimal flows:\\s*\\[([^\\]]*)\\]\',\n        \'min_cost\': r\'Minimal cost:\\s*([-\\d.eE+]+)\',\n        \'n_iter_best\': r\'Obtained after\\s*(\\d+)\\s*iterations\',\n        \'last_kernel\': r\'Last kernel parameters\\s*:\\s*(.+)\',\n        \'strategy\': r\'Optimization strategy:\\s*(.+?)\\.\',\n        \'pca_file\': r\'PCA decomposition file:\\s*(.+)\',\n        \'pca_metric\': r\'PCA cost metric:\\s*(.+)\',\n        \'target_line\': r\'Target spectrum:\\s*(.+)\',\n        \'neg_var\': r\'Predicted negative variance:\\s*(\\w+)\',\n        \'date\': r\'^\\S+:\\s+(\\d{4}-\\d{2}-\\d{2}T[\\d:.]+)\',\n    }\n    for key, pat in pats.items():\n        m = re.search(pat, txt, re.MULTILINE)\n        if m:\n            out[key] = m.group(1).strip()\n    if \'optimal_flows\' in out:\n        out[\'optimal_flows\'] = np.fromstring(out[\'optimal_flows\'], sep=\' \')\n    for k in (\'min_cost\',):\n        if k in out:\n            out[k] = float(out[k])\n    return out\n\n\ndef load_tables(path):\n    path = Path(path)\n    flows = np.loadtxt(path / \'pump_flows.csv\', delimiter=\';\', ndmin=2)\n    costs = np.atleast_1d(np.loadtxt(path / \'costs.csv\'))\n    return flows, costs\n\n\ndef load_gp(path, iteration=None):\n    path = Path(path)\n    files = sorted((path / \'gaussian_processes\').glob(\'gp_model_iter_*.pkl\'))\n    if not files:\n        raise FileNotFoundError(f\'no gp_model_iter_*.pkl in {path}\')\n    if iteration is None:\n        src = files[-1]\n    else:\n        src = path / \'gaussian_processes\' / f\'gp_model_iter_{int(iteration):03d}.pkl\'\n        if not src.exists():\n            raise FileNotFoundError(src)\n    with open(src, \'rb\') as f:\n        d = pickle.load(f)\n    d[\'_file\'] = src.name\n    d[\'_available\'] = [int(re.search(r\'(\\d+)\\.pkl$\', f.name).group(1)) for f in files]\n    return d\n\n\nCFG = load_config(EXPERIMENT_PATH)\nH5M = load_h5_meta(EXPERIMENT_PATH)\nPTX = load_params_txt(EXPERIMENT_PATH)\nFLOWS_CSV, COSTS_CSV = load_tables(EXPERIMENT_PATH)\nif STOP_AT_EVAL is not None:\n    # evaluation e (0-based) is the (e - n_init + 1)-th BO iteration; the\n    # pickle of that iteration is the first one trained on it. Init\n    # evaluations all live in gp_model_iter_000.\n    _e = int(STOP_AT_EVAL) - int(POINT_LABEL_BASE)\n    GP_ITER = max(0, _e - int(CFG.get(\'nb_init_samples\', 0)) + 1)\n    print(f\'STOP_AT_EVAL = {STOP_AT_EVAL} -> GP_ITER = {GP_ITER}\')\nGPD = load_gp(EXPERIMENT_PATH, GP_ITER)\nGP = GPD[\'gaussian_process\']\n\nprint(f"config  : _config.json          ({len(CFG)} keys)")\nprint(f"h5      : {H5M.get(\'_file\', \'none\')}")\nprint(f"params  : {PTX.get(\'_file\', \'none\')}")\nprint(f"gp      : {GPD[\'_file\']}   (available iters {GPD[\'_available\'][0]}..{GPD[\'_available\'][-1]})")\nprint(f"flows   : {FLOWS_CSV.shape}   costs: {COSTS_CSV.shape}")',
    'def _fmt(v):\n    if isinstance(v, (list, tuple, np.ndarray)):\n        a = np.asarray(v)\n        if a.dtype.kind in \'fc\':\n            return \'[\' + \', \'.join(f\'{x:g}\' for x in a.ravel()) + \']\'\n        return \'[\' + \', \'.join(str(x) for x in a.ravel()) + \']\'\n    if isinstance(v, float):\n        return f\'{v:g}\'\n    return str(v)\n\n\n# config key -> (label, h5 attribute used for the cross-check)\nSUMMARY_ROWS = [\n    (\'--- Objective\', None, None),\n    (\'target_style\',            \'Target style\',            \'target_style\'),\n    (\'cost_method\',             \'Cost method\',             \'cost_method\'),\n    (\'pca_cost_metric\',         \'PCA cost metric\',         None),\n    (\'pca_n_components\',        \'PCA components\',          None),\n    (\'cost_transform\',          \'Cost transform\',          None),\n    (\'tg_wl\',                   \'Target wavelength (nm)\',  \'target_wavelength\'),\n    (\'tg_abs\',                  \'Target absorbance\',       \'target_absorbance\'),\n    (\'tol\',                     \'Tolerance\',               \'tolerance\'),\n    (\'rel_tol\',                 \'Relative tolerance\',      None),\n    (\'--- Optimiser\', None, None),\n    (\'acq_func\',                \'Acquisition function\',    \'acquisition_function\'),\n    (\'maxiter\',                 \'Max iterations\',          None),\n    (\'nb_init_samples\',         \'Initial samples\',         None),\n    (\'init_option\',             \'Initial design\',          None),\n    (\'de_maxiter\',              \'DE maxiter\',              None),\n    (\'de_tol\',                  \'DE tolerance\',            None),\n    (\'bo_preset_override\',      \'BO preset\',               None),\n    (\'--- GP model\', None, None),\n    (\'kernel_type_override\',    \'Kernel (requested)\',      None),\n    (\'rbf_length_scale_override\', \'Initial length scale\',  None),\n    (\'length_scale_min\',        \'Length-scale floor\',      None),\n    (\'white_noise_override\',    \'White noise (init)\',      None),\n    (\'white_noise_floor\',       \'White-noise floor\',       None),\n    (\'white_noise_ceiling\',     \'White-noise ceiling\',     None),\n    (\'gp_n_restarts_override\',  \'GP restarts\',             None),\n    (\'--- Robustness\', None, None),\n    (\'flat_strategy_override\',      \'Flat strategy\',        None),\n    (\'flat_penalty_factor_override\', \'Flat penalty factor\', None),\n    (\'flat_nearest_margin_override\', \'Flat nearest margin\', None),\n    (\'flat_nearest_cap_override\',    \'Flat nearest cap\',    None),\n    (\'huber_delta_nm_override\',  \'Huber delta (nm)\',       None),\n    (\'feasibility_aware_override\', \'Feasibility-aware\',    None),\n    (\'use_physics_ghosts\',      \'Virtual observations\',    None),\n    (\'ghost_noise_level\',       \'Virtual obs. noise level\', None),\n    (\'ghost_penalty_lift\',      \'Virtual obs. penalty lift\', None),\n    (\'--- Fluidics\', None, None),\n    (\'number_of_pumps\',         \'Number of pumps\',         None),\n    (\'total_flow\',              \'Total flow\',              \'total_flow\'),\n    (\'reagents\',                \'Reagents\',                \'reagents\'),\n    (\'fixed_pumps\',             \'Fixed pumps\',             None),\n    (\'fixed_flows\',             \'Fixed flows\',             None),\n    (\'residence_time\',          \'Residence time (s)\',      \'residence_time\'),\n    (\'--- Spectrometer\', None, None),\n    (\'spectro_integration_time\', \'Integration time (ms)\',  \'spectro_integration_time_ms\'),\n    (\'spectro_number_average\',  \'Averages\',                \'spectro_number_averages\'),\n    (\'spectro_smoothing\',       \'Smoothing (px)\',          \'spectro_smoothing_px\'),\n    (\'termination_correlation\', \'Termination correlation\', \'termination_correlation_threshold\'),\n    (\'termination_amplitude\',   \'Termination amplitude\',   \'termination_amplitude_threshold\'),\n    (\'termination_timeout\',     \'Termination timeout (s)\', \'termination_timeout_s\'),\n]\n\n\ndef _same(a, b):\n    try:\n        if isinstance(a, (list, np.ndarray)) or isinstance(b, (list, np.ndarray)):\n            aa, bb = np.asarray(a).ravel(), np.asarray(b).ravel()\n            if aa.shape != bb.shape:\n                return False\n            if aa.dtype.kind in \'fiu\' and bb.dtype.kind in \'fiu\':\n                return bool(np.allclose(aa.astype(float), bb.astype(float)))\n            return bool(np.all([str(x).strip() == str(y).strip() for x, y in zip(aa, bb)]))\n        if isinstance(a, (int, float, np.number)) and isinstance(b, (int, float, np.number)):\n            return bool(np.isclose(float(a), float(b)))\n        return str(a).strip() == str(b).strip()\n    except Exception:\n        return False\n\n\ndef build_summary():\n    lines, mismatches = [], []\n    name = EXPERIMENT_PATH.name\n    lines.append(\'=\' * 72)\n    lines.append(f\'  BO RUN  {name}\')\n    if PTX.get(\'date\'):\n        lines.append(f\'  started {PTX["date"]}\')\n    lines.append(\'=\' * 72)\n    for key, label, h5key in SUMMARY_ROWS:\n        if label is None:\n            lines.append(\'\')\n            lines.append(key.replace(\'--- \', \'\').upper())\n            lines.append(\'-\' * 72)\n            continue\n        if key not in CFG:\n            continue\n        val = CFG[key]\n        flag = \'\'\n        if h5key and h5key in H5M:\n            if _same(val, H5M[h5key]):\n                flag = \'   [h5 ok]\'\n            else:\n                flag = f\'   [!! h5 = {_fmt(H5M[h5key])}]\'\n                mismatches.append((label, _fmt(val), _fmt(H5M[h5key])))\n        lines.append(f\'  {label:<26s} {_fmt(val)}{flag}\')\n    return lines, mismatches\n\n\nSUMMARY_LINES, MISMATCHES = build_summary()\nprint(\'\\n\'.join(SUMMARY_LINES))',
    '# ---- GP-side summary: what the model actually ended up with -----------\nGHOST_MASK = None\n\n\ndef gp_training_split():\n    \'\'\'Split gp.X_train_ into real measurements and ghost/penalty points.\n\n    gp.X_train_ columns are the varying pumps (`GPD[\'varying_pumps\']`, not\n    necessarily the first N raw pump columns - fixed pumps can sit anywhere,\n    e.g. PVP/Acetic Acid fixed but AgNO3/Ascorbic Acid/NaBH4 varying).\n    \'\'\'\n    X = np.asarray(GP.X_train_, dtype=float)\n    var_idx = np.flatnonzero(np.asarray(GPD[\'varying_pumps\'], dtype=bool))\n    real = np.asarray(GPD[\'training_flows\'],\n                      dtype=float)[:, var_idx][:, :X.shape[1]]\n    is_ghost = np.ones(len(X), dtype=bool)\n    for i, x in enumerate(X):\n        if np.any(np.all(np.isclose(real, x, atol=1e-4), axis=1)):\n            is_ghost[i] = False\n    return is_ghost\n\n\nGHOST_MASK = gp_training_split()\nN_REAL = int((~GHOST_MASK).sum())\nN_GHOST = int(GHOST_MASK.sum())\n\ngp_lines = []\ngp_lines.append(\'\')\ngp_lines.append(\'FITTED GP  (\' + GPD[\'_file\'] + \')\')\ngp_lines.append(\'-\' * 72)\ngp_lines.append(f\'  {"Iteration":<26s} {GPD.get("iteration")}\')\ngp_lines.append(f\'  {"Kernel (fitted)":<26s} {GP.kernel_}\')\nif PTX.get(\'last_kernel\'):\n    ok = str(PTX[\'last_kernel\']).split(\'+\')[0].strip()[:24] in str(GP.kernel_)\n    gp_lines.append(f\'  {"Kernel (parameters.txt)":<26s} {PTX["last_kernel"]}\'\n                    + (\'   [ok]\' if ok else \'   [!! differs from pickle]\'))\ngp_lines.append(f\'  {"Training points":<26s} {len(GP.X_train_)}  \'\n                f\'({N_REAL} measured + {N_GHOST} virtual)\')\ngp_lines.append(f\'  {"GP input dimension":<26s} {GP.X_train_.shape[1]}\')\ngp_lines.append(f\'  {"normalize_y":<26s} {GP.normalize_y}\')\nalpha = np.atleast_1d(GP.alpha)\ngp_lines.append(f\'  {"alpha (noise)":<26s} \'\n                + (f\'{float(alpha[0]):g}\' if alpha.size == 1\n                   else f\'{float(np.min(alpha)):g} .. {float(np.max(alpha)):g}\'))\ngp_lines.append(f\'  {"log marginal likelihood":<26s} \'\n                f\'{GP.log_marginal_likelihood_value_:.4f}\')\n\ngp_lines.append(\'\')\ngp_lines.append(\'RESULT\')\ngp_lines.append(\'-\' * 72)\n_costs = np.asarray(GPD[\'training_costs\'], dtype=float)\n_best = int(np.argmin(_costs))\ngp_lines.append(f\'  {"Evaluations":<26s} {len(_costs)}  \'\n                f\'({CFG.get("nb_init_samples", "?")} init + \'\n                f\'{len(_costs) - int(CFG.get("nb_init_samples", 0))} BO)\')\ngp_lines.append(f\'  {"Best cost":<26s} {_costs[_best]:.6g}   (evaluation #{_best})\')\ngp_lines.append(f\'  {"Best flows":<26s} \'\n                f\'{np.array2string(np.asarray(GPD["training_flows"])[_best], precision=4)}\')\nif PTX.get(\'result\'):\n    gp_lines.append(f\'  {"Reported outcome":<26s} {PTX["result"]}\')\nif PTX.get(\'min_cost\') is not None:\n    ok = np.isclose(PTX[\'min_cost\'], _costs[_best], rtol=1e-3)\n    gp_lines.append(f\'  {"parameters.txt min cost":<26s} {PTX["min_cost"]:.6g}\'\n                    + (\'   [ok]\' if ok else \'   [!! differs]\'))\nif PTX.get(\'neg_var\'):\n    gp_lines.append(f\'  {"Negative variance seen":<26s} {PTX["neg_var"]}\')\nif PTX.get(\'target_line\'):\n    gp_lines.append(f\'  {"Target spectrum":<26s} {PTX["target_line"]}\')\n\nif MISMATCHES:\n    gp_lines.append(\'\')\n    gp_lines.append(\'!! CROSS-CHECK MISMATCHES config vs H5\')\n    gp_lines.append(\'-\' * 72)\n    for label, a, b in MISMATCHES:\n        gp_lines.append(f\'  {label:<26s} config={a}   h5={b}\')\n\nSUMMARY_LINES += gp_lines\nprint(\'\\n\'.join(gp_lines))\n\n(OUTPUT_DIR / \'summary.txt\').write_text(\'\\n\'.join(SUMMARY_LINES), encoding=\'utf-8\')\nprint(\'\\nsaved ->\', OUTPUT_DIR / \'summary.txt\')',
    'def detect_varying_pumps():\n    n_pumps = int(CFG.get(\'number_of_pumps\', FLOWS_CSV.shape[1]))\n    reagents = [str(r) for r in CFG.get(\'reagents\', [f\'pump {i+1}\' for i in range(n_pumps)])]\n\n    from_cfg = ~np.asarray(CFG.get(\'fixed_pumps\', [False] * n_pumps), dtype=bool)\n    from_std = np.std(FLOWS_CSV, axis=0) > VARYING_STD_EPS\n    from_gp = np.asarray(GPD[\'varying_pumps\'], dtype=bool) if \'varying_pumps\' in GPD else None\n    from_h5 = np.asarray(H5M[\'varying_pumps\'], dtype=bool) if \'varying_pumps\' in H5M else None\n\n    var = from_cfg if VARYING_FROM == \'config\' else from_std\n    print(\'varying-pump detection\')\n    print(f\'  _config.json fixed_pumps -> {from_cfg.astype(int)}\')\n    print(f\'  std(pump_flows) > {VARYING_STD_EPS} -> {from_std.astype(int)}\')\n    if from_gp is not None:\n        print(f\'  gp pickle varying_pumps  -> {from_gp.astype(int)}\'\n              + (\'\' if np.array_equal(from_gp, var) else \'   [!! differs]\'))\n    if from_h5 is not None:\n        print(f\'  h5 varying_pumps         -> {from_h5.astype(int)}\'\n              + (\'\' if np.array_equal(from_h5, var) else \'   [!! differs]\'))\n    if not np.array_equal(from_cfg, from_std):\n        print(\'  [!] config and flow spread disagree - check VARYING_FROM\')\n    return var, reagents\n\n\nVARYING, REAGENTS = detect_varying_pumps()\nVAR_IDX = np.flatnonzero(VARYING)\nN_VAR = int(VARYING.sum())\nN_DIMS = N_VAR - 1                       # GP input dimension\nVAR_NAMES = [REAGENTS[i] for i in VAR_IDX]\nVAR_LABELS = [pretty(n) for n in VAR_NAMES]\nFIXED_IDX = np.flatnonzero(~VARYING)\n\nQTOT = float(CFG.get(\'total_flow\', GPD.get(\'total_var_flow\', 14.0)))\nQTOT_VAR = float(GPD.get(\'total_var_flow\',\n                         QTOT - float(np.sum(np.asarray(CFG.get(\'fixed_flows\', [0])) [FIXED_IDX]))\n                         if FIXED_IDX.size else QTOT))\n\n# measured points, in varying coordinates\nFLOWS_ALL = np.asarray(GPD[\'training_flows\'], dtype=float)\nCOSTS_ALL = np.asarray(GPD[\'training_costs\'], dtype=float)\nCOMP_REAL = FLOWS_ALL[:, VAR_IDX]\nN_INIT = int(CFG.get(\'nb_init_samples\', 0))\nBEST_I = int(np.argmin(COSTS_ALL))\n\n# ghost points, reconstructed from the GP training matrix\nX_TRAIN = np.asarray(GP.X_train_, dtype=float)\nCOMP_GHOST = np.column_stack([X_TRAIN[GHOST_MASK],\n                              QTOT_VAR - X_TRAIN[GHOST_MASK].sum(axis=1)]) \\\n    if N_GHOST else np.zeros((0, N_VAR))\n\nassert GP.X_train_.shape[1] == N_DIMS, (\n    f\'GP expects {GP.X_train_.shape[1]} features but {N_VAR} varying pumps were \'\n    f\'detected -> change VARYING_FROM in the config cell\')\n\nprint()\nprint(f\'varying pumps : {N_VAR}  ->  {VAR_NAMES}\')\nif FIXED_IDX.size:\n    print(f\'fixed pumps   : {[REAGENTS[i] for i in FIXED_IDX]} \'\n          f\'at {np.asarray(CFG.get("fixed_flows"))[FIXED_IDX]}\')\nprint(f\'GP dimension  : {N_DIMS}   (simplex of dimension {N_DIMS})\')\nprint(f\'total flow    : {QTOT}   varying total: {QTOT_VAR}\')\nprint(f\'evaluations   : {len(COSTS_ALL)}  ({N_INIT} init) | \'\n      f\'virtual observations: {N_GHOST}\')',
    '# ---- GP evaluation ----------------------------------------------------\ndef comp_to_feat(comp):\n    \'\'\'(N, N_VAR) varying flows -> (N, N_DIMS) GP features.\'\'\'\n    return np.asarray(comp, dtype=float)[:, :N_DIMS]\n\n\ndef feat_to_comp(feat):\n    \'\'\'(N, N_DIMS) GP features -> (N, N_VAR) varying flows.\'\'\'\n    feat = np.atleast_2d(np.asarray(feat, dtype=float))\n    return np.column_stack([feat, QTOT_VAR - feat.sum(axis=1)])\n\n\ndef gp_eval(comp, chunk=50000):\n    \'\'\'Posterior mean and std at varying-flow compositions.\'\'\'\n    X = comp_to_feat(comp)\n    mus, sds = [], []\n    for i in range(0, len(X), chunk):\n        m, s = GP.predict(X[i:i + chunk], return_std=True)\n        mus.append(m)\n        sds.append(s)\n    return np.concatenate(mus), np.concatenate(sds)\n\n\ndef feasible(comp, tol=1e-9):\n    comp = np.asarray(comp, dtype=float)\n    return np.all(comp >= -tol, axis=1)\n\n\ndef sample_simplex(n):\n    \'\'\'Uniform sample of the varying-flow simplex.\'\'\'\n    return RNG.dirichlet(np.ones(N_VAR), size=n) * QTOT_VAR\n\n\ndef simplex_lattice(scale, dim):\n    \'\'\'Integer compositions of `scale` into `dim` parts (regular lattice).\'\'\'\n    if dim == 1:\n        return np.array([[scale]])\n    out = []\n    for i in range(scale + 1):\n        sub = simplex_lattice(scale - i, dim - 1)\n        out.append(np.column_stack([np.full(len(sub), i), sub]))\n    return np.vstack(out)\n\n\n# ---- barycentric projections -----------------------------------------\n_SQ3 = np.sqrt(3.0) / 2.0\nTRI_XY = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, _SQ3]])          # L, R, T\nTET_XYZ = np.array([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0],\n                    [0.5, _SQ3, 0.0],\n                    [0.5, np.sqrt(3) / 6, np.sqrt(2.0 / 3.0)]])\nTET_EDGES = [(0, 1), (0, 2), (1, 2), (0, 3), (1, 3), (2, 3)]\nTET_FACES = [(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]\n\n\ndef bary3(w):\n    \'\'\'(N,3) weights summing to 1 -> (N,2) cartesian.\'\'\'\n    w = np.atleast_2d(np.asarray(w, dtype=float))\n    return w @ TRI_XY\n\n\ndef bary4(w):\n    \'\'\'(N,4) weights summing to 1 -> (N,3) cartesian.\'\'\'\n    w = np.atleast_2d(np.asarray(w, dtype=float))\n    return w @ TET_XYZ\n\n\ndef normalise(w):\n    w = np.atleast_2d(np.asarray(w, dtype=float))\n    s = w.sum(axis=1, keepdims=True)\n    return np.divide(w, s, out=np.full_like(w, np.nan), where=s > 1e-12)\n\n\n# ---- min-projection ("best slice") -----------------------------------\ndef min_projection(comp, mu, sd, keep, nbins, centers=True,\n                   return_argmin=False):\n    \'\'\'Collapse the axes not in `keep` by keeping the *best* (lowest mu) point.\n\n    The kept composition is renormalised to sum to 1, binned on a regular\n    lattice, and the lowest-mu sample of each cell wins. Returns the cell\n    positions (barycentric, summing to 1) with the winning mu and sd.\n\n    `return_argmin` appends the *collapsed* flow rates of that winning sample,\n    one column per dropped reagent. That is the dimension a projection throws\n    away - mapping it recovers "how much of the dropped reagent it takes to\n    reach this best cost", which is the part of a 4-D posterior a 3-D drawing\n    cannot show.\n    \'\'\'\n    keep = list(keep)\n    comp = np.asarray(comp, dtype=float)\n    w = normalise(comp[:, keep])\n    ok = np.isfinite(w).all(axis=1)\n    w, mu, sd, comp = w[ok], np.asarray(mu)[ok], np.asarray(sd)[ok], comp[ok]\n    drop = [i for i in range(comp.shape[1]) if i not in keep]\n    key = np.clip((w[:, :-1] * nbins).astype(np.int64), 0, nbins - 1)\n    flat = key @ (nbins ** np.arange(key.shape[1], dtype=np.int64))\n    order = np.lexsort((mu, flat))\n    fs = flat[order]\n    first = np.ones(fs.size, dtype=bool)\n    first[1:] = fs[1:] != fs[:-1]\n    sel = order[first]\n    if not centers:\n        out = (w[sel], mu[sel], sd[sel])\n        return out + (comp[sel][:, drop],) if return_argmin else out\n    c = (key[sel] + 0.5) / nbins\n    last = 1.0 - c.sum(axis=1)\n    good = last > -0.5 / nbins\n    wc = normalise(np.column_stack([c[good], np.clip(last[good], 0, None)]))\n    out = (wc, mu[sel][good], sd[sel][good])\n    if return_argmin:\n        out = out + (comp[sel][good][:, drop],)\n    return out\n\n\ndef axis_influence(n=120000, nbins=24):\n    \'\'\'Rank the varying reagents by how much the *best achievable* cost moves.\'\'\'\n    comp = sample_simplex(n)\n    mu, sd = gp_eval(comp)\n    score = np.zeros(N_VAR)\n    for i in range(N_VAR):\n        b = np.clip((comp[:, i] / QTOT_VAR * nbins).astype(int), 0, nbins - 1)\n        curve = np.full(nbins, np.nan)\n        for k in range(nbins):\n            m = b == k\n            if m.sum() > 5:\n                curve[k] = mu[m].min()\n        score[i] = np.nanmax(curve) - np.nanmin(curve)\n    return score\n\n\nINFLUENCE = axis_influence() if N_DIMS >= 3 else np.zeros(N_VAR)\nif N_DIMS >= 3:\n    rank = np.argsort(-INFLUENCE)\n    print(\'axis influence (spread of the best achievable cost):\')\n    for i in rank:\n        print(f\'  {VAR_NAMES[i]:<16s} {INFLUENCE[i]:.4g}\')\n    KEEP4 = sorted(rank[:4]) if TETRA_KEEP is None else list(TETRA_KEEP)\n    KEEP3 = sorted(rank[:3]) if TERNARY_KEEP is None else list(TERNARY_KEEP)\n    SLICE_DIM = int(rank[-1]) if SLICE_DIM_5D is None else int(SLICE_DIM_5D)\nelse:\n    KEEP4, KEEP3, SLICE_DIM = list(range(min(4, N_VAR))), list(range(min(3, N_VAR))), 0\nprint(\'OK\')',
    'def load_spectra():\n    \'\'\'Final spectrum of every evaluation, in evaluation order.\n\n    `lambda_max`/`abs_max` come from the **live** `spectra/lambda_max` and\n    `spectra/extinction_max` arrays (last sample) - the raw per-timestamp peak the\n    acquisition pipeline actually recorded, which is what fed the cost\n    function. `result@lambda_max_nm` is a *post-hoc* argmax of the\n    averaged steady-state spectrum and can differ by several nm; using it\n    here would make the reported cost look inconsistent with `costs.csv`.\n    \'\'\'\n    d = Path(EXPERIMENT_PATH) / \'samples\'\n    if not (HAS_H5 and d.is_dir()):\n        return None\n    items = []\n    for f in d.glob(\'*.h5\'):\n        m = re.search(r\'_(init|iter)_(\\d+)\\.h5$\', f.name)\n        if m:\n            items.append(((0 if m.group(1) == \'init\' else 1,\n                           int(m.group(2))), f))\n    if not items:\n        return None\n    items.sort()\n    wl, spec, amax, lmax, names = None, [], [], [], []\n    for key, f in items:\n        with h5py.File(f, \'r\') as h:\n            if \'result/extinction_final\' not in h:\n                return None\n            spec.append(np.asarray(h[\'result/extinction_final\'][()], dtype=float))\n            st = h[\'result\'].attrs\n            if \'spectra/lambda_max\' in h and len(h[\'spectra/lambda_max\']):\n                amax.append(float(np.asarray(h[\'spectra/extinction_max\'][()])[-1]))\n                lmax.append(float(np.asarray(h[\'spectra/lambda_max\'][()])[-1]))\n            else:\n                amax.append(float(st.get(\'extinction_max\', np.nan)))\n                lmax.append(float(st.get(\'lambda_max_nm\', np.nan)))\n            if wl is None:\n                wl = np.asarray(h[\'spectra/wavelength\'][()], dtype=float)\n        names.append((\'init_\' if key[0] == 0 else \'iter_\') + str(key[1]))\n    return dict(wl=wl, spectra=np.vstack(spec), abs_max=np.array(amax),\n                lambda_max=np.array(lmax), names=names)\n\n\ndef load_target_spectrum():\n    \'\'\'Target spectrum: H5 first, then target_spectrum.csv.\'\'\'\n    d = Path(EXPERIMENT_PATH) / \'samples\'\n    if HAS_H5 and d.is_dir():\n        for f in sorted(d.glob(\'*.h5\')):\n            with h5py.File(f, \'r\') as h:\n                g = \'run_settings/target_spectrum\'\n                if g + \'/extinction\' in h:\n                    return (np.asarray(h[g + \'/wavelength\'][()], dtype=float),\n                            np.asarray(h[g + \'/extinction\'][()], dtype=float))\n    p = Path(EXPERIMENT_PATH) / \'target_spectrum.csv\'\n    if p.exists():\n        a = np.loadtxt(p, delimiter=\';\')\n        a = np.atleast_2d(a)\n        if a.shape[0] == 2:            # wavelengths row + absorbance row\n            return a[0], a[1]\n        return np.arange(a.shape[-1], dtype=float), a.ravel()\n    return None, None\n\n\ndef load_measurement_last_row(path):\n    \'\'\'Last raw row of a measurement_spectra_*.csv (live steady-state reading).\n\n    Used for PCA-target cost reproduction: for S19-M04, the mean-of-tail\n    \'result/extinction_final\' happened to already equal the last row, but the\n    target spectrum lives outside any run folder (no H5), so it is read from\n    its raw CSV directly, the same way.\n    \'\'\'\n    path = Path(path)\n    with open(path, encoding=\'utf-8\') as f:\n        header = f.readline().rstrip(\'\\n\').split(\';\')\n    wl = np.array([float(x) for x in header[1:]], dtype=float)\n    data = np.genfromtxt(path, delimiter=\';\', skip_header=1, dtype=str)\n    if data.ndim == 1:\n        data = data.reshape(1, -1)\n    return wl, data[-1, 1:].astype(float)\n\n\ndef load_pca_model():\n    \'\'\'Load a PCA decomposition .npz - two key schemas seen in the wild:\n    {wl, mean, comps, explained_variance, normalize} (e.g. some copies of\n    pca_model_norm6.npz) and {wavelengths, mean, components, ...} (e.g.\n    pca_model_5.npz, and other copies of pca_model_norm6.npz) - same content,\n    different names, so both are supported.\n    \'\'\'\n    if not PCA_MODEL_PATH:\n        return None\n    d = np.load(PCA_MODEL_PATH, allow_pickle=True)\n    wl_key = \'wl\' if \'wl\' in d.files else \'wavelengths\'\n    comps_key = \'comps\' if \'comps\' in d.files else \'components\'\n    return dict(wl=np.asarray(d[wl_key], dtype=float),\n                mean=np.asarray(d[\'mean\'], dtype=float),\n                comps=np.asarray(d[comps_key], dtype=float),\n                explained_variance=np.asarray(d[\'explained_variance\'], dtype=float),\n                normalize=str(d[\'normalize\']))\n\n\ndef pca_n_components(model):\n    if PCA_N_COMPONENTS is not None:\n        return int(PCA_N_COMPONENTS)\n    n = CFG.get(\'pca_n_components\')\n    return int(n) if n else model[\'comps\'].shape[0]\n\n\ndef pca_metric():\n    if PCA_COST_METRIC is not None:\n        return str(PCA_COST_METRIC).lower()\n    return str(CFG.get(\'pca_cost_metric\', \'l2\')).lower()\n\n\ndef pca_score(wl, spectrum, model):\n    \'\'\'Project a raw spectrum onto the PCA model\'s score space.\n\n    Confirmed against costs.csv: interpolate onto the model\'s own wavelength\n    grid, normalize by the spectrum\'s own max only if model[\'normalize\'] ==\n    \'max\' (some models use \'none\' - no normalization at all), subtract the\n    model mean, project onto pca_n_components() components - no variance\n    weighting.\n    \'\'\'\n    interp = np.interp(model[\'wl\'], wl, spectrum)\n    if model[\'normalize\'] == \'max\':\n        interp = interp / interp.max()\n    n = pca_n_components(model)\n    return (interp - model[\'mean\']) @ model[\'comps\'][:n].T\n\n\ndef cost_descr():\n    \'\'\'Human-readable cost definition for axis labels.\n\n    `cost_method` in _config.json is only the lambda_max / full_spectrum\n    metric: a PCA target uses `pca_cost_metric` instead, and printing "L2"\n    on a cosine-cost run mislabels the whole convergence figure.\n    \'\'\'\n    if TARGET_STYLE == \'pca\':\n        metric = {\'cosine\': \'cosine distance\', \'l2\': \'L2 distance\'}.get(\n            pca_metric(), pca_metric())\n        n = CFG.get(\'pca_n_components\')\n        return (f\'{metric} in PCA space\' + (f\', {int(n)} components\' if n\n                                            else \'\'))\n    m = str(CFG.get(\'cost_method\', \'\')).upper()\n    if TARGET_STYLE == \'lambda_max\':\n        return f\'{m} on $\\\\lambda_{{max}}$\' if m else \'on $\\\\lambda_{{max}}$\'\n    return m or \'cost\'\n\n\ndef pca_cost(score_a, score_b, metric=None):\n    a, b = np.asarray(score_a), np.asarray(score_b)\n    metric = metric or pca_metric()\n    if metric == \'cosine\':\n        return float(1.0 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))\n    return float(np.sum((a - b) ** 2))\n\n\nSPEC = load_spectra()\nTARGET_WL, TARGET_ABS = load_target_spectrum()\n\nTARGET_STYLE = str(CFG.get(\'target_style\', \'\')).lower()\nPCA_MODEL = load_pca_model() if TARGET_STYLE == \'pca\' else None\nPCA_TARGET_WL = PCA_TARGET_SPEC = None\nif TARGET_STYLE == \'pca\' and PCA_TARGET_PATH:\n    PCA_TARGET_WL, PCA_TARGET_SPEC = load_measurement_last_row(PCA_TARGET_PATH)\n\nFLAT_THR = float(CFG.get(\'threshold\', 0.01) if FLAT_STD_THRESHOLD is None\n                 else FLAT_STD_THRESHOLD)\nIS_FLAT = np.zeros(len(COSTS_ALL), dtype=bool)\nSPEC_OK = False\nif (SPEC is not None and GP_ITER is not None\n        and len(SPEC[\'spectra\']) > len(COSTS_ALL)):\n    # an earlier GP snapshot: keep the spectra of the evaluations it saw\n    # (they are in evaluation order), drop the later ones\n    _n = len(COSTS_ALL)\n    SPEC = dict(SPEC, spectra=SPEC[\'spectra\'][:_n], abs_max=SPEC[\'abs_max\'][:_n],\n                lambda_max=SPEC[\'lambda_max\'][:_n], names=SPEC[\'names\'][:_n])\n    print(f\'GP snapshot {GPD["_file"]}: spectra truncated to the first {_n} \'\n          f\'evaluations\')\nif SPEC is not None and len(SPEC[\'spectra\']) == len(COSTS_ALL):\n    SPEC_OK = True\n    IS_FLAT = SPEC[\'spectra\'].std(axis=1) < FLAT_THR\nelif SPEC is not None:\n    print(f\'[!] {len(SPEC["spectra"])} spectra for {len(COSTS_ALL)} costs \'\n          f\'-> spectra ignored, no flat detection\')\nelse:\n    print(\'[!] no samples/*.h5 -> no flat detection, no spectrum figure\')\n\nif TARGET_STYLE == \'pca\':\n    pca_lines = [\'\', \'PCA TARGET\', \'-\' * 72]\n    if PCA_MODEL is None:\n        pca_lines.append(\'  target_style = pca but PCA_MODEL_PATH is not set -> \'\n                         \'the best-spectrum figure will show a plain overlay, \'\n                         \'NOT the actual PCA-space cost (see _parameters.txt \'\n                         \'for the "PCA decomposition file" path)\')\n    elif PCA_TARGET_SPEC is None:\n        pca_lines.append(f\'  {"PCA model":<26s} {PCA_MODEL_PATH}\')\n        pca_lines.append(\'  PCA_TARGET_PATH is not set -> cannot cross-check \'\n                         \'the PCA cost against costs.csv\')\n    elif SPEC_OK:\n        n_used = pca_n_components(PCA_MODEL)\n        pca_lines.append(f\'  {"PCA model":<26s} {PCA_MODEL_PATH}\')\n        pca_lines.append(f\'  {"PCA target":<26s} {PCA_TARGET_PATH}\')\n        pca_lines.append(f\'  {"Normalize":<26s} {PCA_MODEL["normalize"]}\')\n        pca_lines.append(f\'  {"Metric":<26s} {pca_metric()}\')\n        pca_lines.append(f\'  {"Components used":<26s} {n_used} / \'\n                         f\'{PCA_MODEL["comps"].shape[0]} available\')\n        tgt_score = pca_score(PCA_TARGET_WL, PCA_TARGET_SPEC, PCA_MODEL)\n        recomputed = np.array([\n            pca_cost(pca_score(SPEC[\'wl\'], s, PCA_MODEL), tgt_score)\n            for s in SPEC[\'spectra\']])\n        keep = ~IS_FLAT[:len(recomputed)]   # flat costs are capped, not raw\n        if keep.any():\n            ratio = COSTS_ALL[:len(recomputed)][keep] / np.where(\n                recomputed[keep] == 0, np.nan, recomputed[keep])\n            pca_lines.append(f\'  {"Best-point recomputed":<26s} \'\n                             f\'{recomputed[BEST_I]:.6g}  (costs.csv = \'\n                             f\'{COSTS_ALL[BEST_I]:.6g})\')\n            pca_lines.append(f\'  {"Reproduction (non-flat)":<26s} \'\n                             f\'ratio {np.nanmean(ratio):.4f} +/- \'\n                             f\'{np.nanstd(ratio):.4f} over {keep.sum()} points\')\n    SUMMARY_LINES += pca_lines\n    print(\'\\n\'.join(pca_lines))',
    'def remap_flat_costs(costs, is_flat, X):\n    \'\'\'Cost the flat points would get through the run\'s flat strategy.\n\n    Mirrors production `_apply_flat_strategy`. `X` is the GP feature matrix of\n    the measurements (distances are taken there).\n    \'\'\'\n    y = np.array(costs, dtype=float)\n    strat = (CFG.get(\'flat_strategy_override\', \'nearest_high\')\n             if FLAT_STRATEGY is None else FLAT_STRATEGY)\n    factor = float(CFG.get(\'flat_penalty_factor_override\', 2.0)\n                   if FLAT_PENALTY_FACTOR is None else FLAT_PENALTY_FACTOR)\n    margin = float(CFG.get(\'flat_nearest_margin_override\', 1.25)\n                   if FLAT_NEAREST_MARGIN is None else FLAT_NEAREST_MARGIN)\n    cap_f = float(CFG.get(\'flat_nearest_cap_override\', 3.0)\n                  if FLAT_NEAREST_CAP is None else FLAT_NEAREST_CAP)\n    if not is_flat.any():\n        return y, strat\n    ri = np.flatnonzero(~is_flat)\n    fi = np.flatnonzero(is_flat)\n    if ri.size == 0:\n        return y, strat\n    rc = y[ri]\n\n    if strat == \'penalty_moderate\':\n        y[fi] = factor * float(np.mean(rc))\n    elif strat == \'penalty_max\':\n        y[fi] = factor * float(np.max(rc))\n    elif strat == \'exclude\':\n        y[fi] = np.nan\n    else:                                     # \'nearest_high\' (production)\n        cap = cap_f * float(np.max(rc))\n        if FLAT_NEAREST_RADIUS is not None:\n            radius = float(FLAT_NEAREST_RADIUS)\n        elif len(X) >= 2:\n            dm = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(-1))\n            np.fill_diagonal(dm, np.inf)\n            radius = 1.5 * float(np.median(dm.min(axis=1)))\n        else:\n            radius = np.inf\n        for f in fi:\n            dd = np.sqrt(((X[ri] - X[f]) ** 2).sum(-1))\n            loc = rc[dd <= radius]\n            val = (margin * float(np.max(loc)) if loc.size\n                   else float(np.percentile(rc, 90)))\n            y[f] = min(val, cap)\n    return y, strat\n\n\nCOSTS_PLOT, FLAT_STRAT_USED = remap_flat_costs(\n    COSTS_ALL, IS_FLAT, comp_to_feat(COMP_REAL))\nN_FLAT = int(IS_FLAT.sum())\n\nflat_lines = [\'\', \'FLAT MEASUREMENTS\', \'-\' * 72,\n              f\'  {"Test":<26s} std(absorbance) < {FLAT_THR:g}\',\n              f\'  {"Detected":<26s} {N_FLAT} / {len(COSTS_ALL)}\',\n              f\'  {"Remap strategy":<26s} {FLAT_STRAT_USED} (display only)\']\nfor i in np.flatnonzero(IS_FLAT):\n    nm = SPEC[\'names\'][i] if SPEC_OK else f\'#{i}\'\n    flat_lines.append(f\'    #{i:<3d} {nm:<10s} std={SPEC["spectra"][i].std():.5f}\'\n                      f\'  cost {COSTS_ALL[i]:.5g} -> {COSTS_PLOT[i]:.5g}\')\nSUMMARY_LINES += flat_lines\n(OUTPUT_DIR / \'summary.txt\').write_text(\'\\n\'.join(SUMMARY_LINES), encoding=\'utf-8\')\nprint(\'\\n\'.join(flat_lines))',
    'C_INIT = \'#b0b0b0\'\nC_BO = \'#1f77b4\'\nC_BEST = \'#e8261a\'\nC_GHOST = \'#7a7a7a\'\nC_FLAT = \'#d95f02\'\nC_TARGET = \'#7b3294\'\n\n\ndef save_fig(fig, name):\n    paths = []\n    if SAVE_PNG:\n        p = OUTPUT_DIR / f\'{name}.png\'\n        fig.savefig(p, dpi=mpl.rcParams[\'savefig.dpi\'], bbox_inches=\'tight\',\n                    facecolor=\'white\')\n        print(\'saved ->\', p)\n        paths.append(p)\n    for fmt in SAVE_FORMATS:\n        p = OUTPUT_DIR / f\'{name}.{fmt}\'\n        kw = dict(bbox_inches=\'tight\', facecolor=\'white\')\n        if fmt == \'tiff\':\n            kw.update(dpi=TIFF_DPI, pil_kwargs={\'compression\': \'tiff_lzw\'})\n        fig.savefig(p, format=fmt, **kw)\n        print(\'saved ->\', p)\n        paths.append(p)\n    return paths[0] if paths else None\n\n\ndef save_html(fig, name):\n    if not (SAVE_HTML and HAS_PLOTLY):\n        return None\n    p = OUTPUT_DIR / f\'{name}.html\'\n    fig.write_html(str(p), include_plotlyjs=\'cdn\')\n    print(\'saved ->\', p)\n    return p\n\n\ndef add_colorbar(fig, mappable, ax, label, pad=0.02, shrink=0.86,\n                 fraction=0.05):\n    cb = fig.colorbar(mappable, ax=ax, pad=pad, shrink=shrink,\n                      fraction=fraction)\n    cb.set_label(label, fontsize=19, labelpad=12)\n    cb.ax.tick_params(labelsize=15, width=1.4, length=6)\n    cb.outline.set_linewidth(1.4)\n    try:                     # no "+5.55e-1" offset headers on flat fields\n        cb.formatter.set_useOffset(False)\n        cb.update_ticks()\n    except AttributeError:\n        pass\n    return cb\n\n\n# ---------------------------------------------------------------- ternary\ndef draw_ternary_axes(ax, labels, nticks=5, lw=2.0, gridcolor=\'0.75\',\n                      label_pad=0.06, fontsize=20, ticksize=18,\n                      flow_scale=None):\n    \'\'\'Triangle frame + gridlines + tick labels. `labels` = (left, right, top).\n\n    `flow_scale`: if set, ticks are printed as real flow rates\n    (`t * flow_scale`, in `FLOW_UNIT`) instead of bare fractions.\n    \'\'\'\n    tri = np.vstack([TRI_XY, TRI_XY[0]])\n    ax.plot(tri[:, 0], tri[:, 1], color=\'black\', lw=lw, zorder=6,\n            solid_capstyle=\'round\')\n    for t in np.linspace(0, 1, nticks + 1)[1:-1]:\n        segs = [\n            ([t, 1 - t, 0], [t, 0, 1 - t]),        # constant left\n            ([1 - t, t, 0], [0, t, 1 - t]),        # constant right\n            ([1 - t, 0, t], [0, 1 - t, t]),        # constant top\n        ]\n        for a, b in segs:\n            p = bary3(np.array([a, b]))\n            ax.plot(p[:, 0], p[:, 1], color=gridcolor, lw=0.9, ls=\':\',\n                    zorder=1)\n        # tick labels\n        tlab = f\'{t * flow_scale:.1f}\' if flow_scale else f\'{t:.1f}\'\n        pb = bary3(np.array([[1 - t, t, 0]]))[0]          # bottom edge\n        ax.text(pb[0], pb[1] - 0.028, tlab, ha=\'center\', va=\'top\',\n                fontsize=ticksize, color=\'0.25\')\n        pr = bary3(np.array([[0, 1 - t, t]]))[0]          # right edge\n        ax.text(pr[0] + 0.022, pr[1] + 0.011, tlab, ha=\'left\',\n                va=\'center\', fontsize=ticksize, color=\'0.25\')\n        pl = bary3(np.array([[t, 0, 1 - t]]))[0]          # left edge\n        ax.text(pl[0] - 0.022, pl[1] + 0.011, tlab, ha=\'right\',\n                va=\'center\', fontsize=ticksize, color=\'0.25\')\n    ax.text(TRI_XY[0, 0] - 0.03, -label_pad, labels[0], ha=\'center\', va=\'top\',\n            fontsize=fontsize, fontweight=\'bold\')\n    ax.text(TRI_XY[1, 0] + 0.03, -label_pad, labels[1], ha=\'center\', va=\'top\',\n            fontsize=fontsize, fontweight=\'bold\')\n    ax.text(TRI_XY[2, 0], TRI_XY[2, 1] + 0.05, labels[2], ha=\'center\',\n            va=\'bottom\', fontsize=fontsize, fontweight=\'bold\')\n    ax.set_xlim(-0.20, 1.20)\n    ax.set_ylim(-0.09, _SQ3 + 0.09)\n    ax.set_aspect(\'equal\', anchor=\'N\')   # letterbox slack goes to the bottom,\n                                         # not between the title and the plot\n    ax.axis(\'off\')\n\n\n# ---------------------------------------------------------------- tetrahedron\ndef view_dir(ax):\n    \'\'\'Unit vector pointing from the scene towards the camera.\'\'\'\n    e, a = np.radians(ax.elev), np.radians(ax.azim)\n    return np.array([np.cos(e) * np.cos(a), np.cos(e) * np.sin(a), np.sin(e)])\n\n\ndef tetra_visibility(ax):\n    \'\'\'(front_face, hidden_edges, silhouette_edges) for the current camera.\n\n    Exact for a convex solid under orthographic projection: a face is visible\n    when its outward normal points towards the camera, an edge is hidden when\n    *both* its faces are back-facing, and it is on the silhouette when it\n    borders one of each. A depth-of-vertex heuristic gets this wrong whenever\n    the outline is a quadrilateral rather than a triangle, which happens over\n    a wide band of camera angles.\n    \'\'\'\n    v = view_dir(ax)\n    centre = TET_XYZ.mean(axis=0)\n    front = []\n    for f in TET_FACES:\n        pts = TET_XYZ[list(f)]\n        n = np.cross(pts[1] - pts[0], pts[2] - pts[0])\n        n = n * np.sign(n @ (pts[0] - centre))          # outward normal\n        front.append(bool(n @ v > 0))\n    hidden, silhouette = [], []\n    for i, j in TET_EDGES:\n        vis = [front[k] for k, f in enumerate(TET_FACES) if i in f and j in f]\n        if not any(vis):\n            hidden.append((i, j))\n        elif not all(vis):\n            silhouette.append((i, j))\n    return front, hidden, silhouette\n\n\ndef _label_tetra_vertices(ax, labels, fontsize, offset=52):\n    \'\'\'Vertex labels pushed radially outward *in screen space*, so they never\n    land on a projected edge whatever the camera angle.\'\'\'\n    try:\n        from mpl_toolkits.mplot3d import proj3d\n        M = ax.get_proj()\n        pts = np.array([proj3d.proj_transform(v[0], v[1], v[2], M)[:2]\n                        for v in TET_XYZ])\n        centre = pts.mean(axis=0)\n        for p, lab in zip(pts, labels):\n            d = p - centre\n            d = d / max(np.linalg.norm(d), 1e-9)\n            ax.annotate(lab, xy=(p[0], p[1]), xycoords=\'data\',\n                        xytext=offset * d, textcoords=\'offset points\',\n                        ha=\'center\', va=\'center\', fontsize=fontsize,\n                        fontweight=\'bold\', annotation_clip=False, zorder=12)\n        return\n    except Exception:\n        pass\n    centre = TET_XYZ.mean(axis=0)                      # fallback: 3-D text\n    for v, lab in zip(TET_XYZ, labels):\n        d = (v - centre) / np.linalg.norm(v - centre)\n        p = v + 0.17 * d\n        ax.text(p[0], p[1], p[2], lab, ha=\'center\', va=\'center\', zorder=6,\n                fontsize=fontsize, fontweight=\'bold\')\n\n\ndef _label_tetra_ticks(ax, flow_scale, fontsize=11, ts=(0.25, 0.5, 0.75),\n                       edges=None):\n    \'\'\'Small flow-rate tick labels along the tetrahedron edges (screen-space\n    offset, same trick as the vertex labels, so they read cleanly at any\n    camera angle) - the scale reference the plain vertex labels don\'t give.\n\n    `edges`: which edges carry labels. Labelling all 6 puts 18 numbers on the\n    figure, and a reader cannot tell which edge any of them belongs to; the\n    silhouette edges alone give the same scale unambiguously.\n    \'\'\'\n    if not flow_scale or TETRA_TICK_EDGES == \'none\':\n        return\n    if edges is None:\n        edges = TET_EDGES\n    try:\n        from mpl_toolkits.mplot3d import proj3d\n        M = ax.get_proj()\n\n        def proj(v):\n            return np.array(proj3d.proj_transform(v[0], v[1], v[2], M)[:2])\n\n        centre2d = proj(TET_XYZ.mean(axis=0))\n        for i, j in edges:\n            for t in ts:\n                v3 = TET_XYZ[i] + t * (TET_XYZ[j] - TET_XYZ[i])\n                p = proj(v3)\n                d = p - centre2d\n                d = d / max(np.linalg.norm(d), 1e-9)\n                ax.annotate(f\'{t * flow_scale:.1f}\', xy=(p[0], p[1]),\n                            xycoords=\'data\', xytext=14 * d,\n                            textcoords=\'offset points\', ha=\'center\',\n                            va=\'center\', fontsize=fontsize, color=\'0.4\',\n                            annotation_clip=False, zorder=5)\n    except Exception:\n        pass\n\n\ndef draw_tetra_axes(ax, labels, lw=2.2, fontsize=19, pad=0.17, flow_scale=None):\n    \'\'\'Tetrahedron frame with hidden-line removal, back-face shading and\n    silhouette-only ticks - the three things that turn the wireframe from an\n    ambiguous flat triangle into a solid.\'\'\'\n    ax.set_xlim(-0.16, 1.16)\n    ax.set_ylim(-0.12, _SQ3 + 0.12)\n    ax.set_zlim(-0.10, TET_XYZ[3, 2] + 0.12)\n    ax.set_axis_off()\n    if TETRA_PERSPECTIVE:\n        try:\n            ax.set_proj_type(\'persp\', focal_length=float(TETRA_PERSPECTIVE))\n        except TypeError:                # matplotlib < 3.6\n            ax.set_proj_type(\'persp\')\n    else:\n        ax.set_proj_type(\'ortho\')\n    ax.computed_zorder = False           # honour our explicit zorders\n    try:\n        ax.set_box_aspect((1.32, _SQ3 + 0.24, TET_XYZ[3, 2] + 0.22), zoom=1.35)\n    except TypeError:\n        ax.set_box_aspect((1.32, _SQ3 + 0.24, TET_XYZ[3, 2] + 0.22))\n\n    front, hidden, silhouette = tetra_visibility(ax)\n    if TETRA_BACKFACE_ALPHA:             # translucent back faces = enclosure\n        back = [TET_XYZ[list(f)] for k, f in enumerate(TET_FACES)\n                if not front[k]]\n        if back:\n            ax.add_collection3d(Poly3DCollection(\n                back, facecolor=\'0.45\', alpha=float(TETRA_BACKFACE_ALPHA),\n                edgecolor=\'none\', zorder=0))\n\n    def _edges(pairs, **kw):\n        if pairs:\n            ax.add_collection3d(Line3DCollection(\n                [(TET_XYZ[i], TET_XYZ[j]) for i, j in pairs], **kw))\n\n    if TETRA_HIDDEN_LINES:\n        _edges(hidden, colors=\'0.62\', linewidths=lw * 0.5,\n               linestyles=[(0, (4.5, 3.5))], zorder=1)\n        _edges([e for e in TET_EDGES if e not in hidden], colors=\'black\',\n               linewidths=lw, zorder=6)\n    else:\n        _edges(TET_EDGES, colors=\'black\', linewidths=lw, zorder=4)\n\n    _label_tetra_vertices(ax, labels, fontsize)   # after the projection is set\n    _label_tetra_ticks(ax, flow_scale,\n                       edges=(silhouette if TETRA_TICK_EDGES == \'front\'\n                              else TET_EDGES))\n\n\ndef plot_tetra_shadows(ax, xyz, color=\'0.55\'):\n    \'\'\'Drop lines to the base face (z = 0, the AgNO3/PVP/Ascorbic triangle)\n    plus a flattened ground shadow. Without this a marker floating inside the\n    tetrahedron has no readable position: the same pixel corresponds to a\n    whole ray through the volume.\'\'\'\n    xyz = np.atleast_2d(np.asarray(xyz, dtype=float))\n    if not len(xyz):\n        return\n    for x, y, z in xyz:\n        ax.plot([x, x], [y, y], [0.0, z], color=color, lw=0.8, ls=\':\',\n                zorder=3, solid_capstyle=\'butt\')\n    ax.scatter(xyz[:, 0], xyz[:, 1], np.zeros(len(xyz)), s=55, c=\'0.72\',\n               marker=\'o\', linewidths=0, alpha=0.55, zorder=2,\n               depthshade=False)\n\n\ndef depth_sort(pts, ax):\n    \'\'\'Order indices back-to-front for the current 3-D view.\'\'\'\n    e, a = np.radians(ax.elev), np.radians(ax.azim)\n    view = np.array([np.cos(e) * np.cos(a), np.cos(e) * np.sin(a), np.sin(e)])\n    return np.argsort(np.asarray(pts) @ view)\n\n\ndef alpha_ramp(v, lo=0.012, hi=0.55, gamma=3.0, mode=\'low\'):\n    \'\'\'Opacity that highlights the interesting part of a scalar field.\n\n    mode=\'low\'  : low values opaque, fading to `lo` at the max (good for a\n                  field with one interesting extreme, e.g. minimum cost).\n    mode=\'high\' : the reverse (used for sigma - high uncertainty opaque).\n    mode=\'deviation\' : opacity ~ |v - mean(v)|, so points near the *typical*\n                  value fade to `lo` and BOTH tails (low-cost basin and\n                  high-cost regions) stand out - useful when most of a 4D+\n                  cloud sits at a similar mid-range cost and a plain low/high\n                  ramp still looks like a nearly solid blob.\n    \'\'\'\n    v = np.asarray(v, dtype=float)\n    if mode == \'deviation\':\n        dev = np.abs(v - np.mean(v))\n        r = dev.max()\n        t = dev / r if r > 0 else np.zeros_like(v)\n        return lo + (hi - lo) * t ** gamma\n    r = np.ptp(v)\n    t = (v - v.min()) / r if r > 0 else np.zeros_like(v)\n    if mode == \'low\':\n        t = 1.0 - t                      # low value -> opaque\n    return lo + (hi - lo) * t ** gamma\n\n\ndef to_color_scale(values):\n    \'\'\'log10(values) when COLOR_SCALE == \'log\' (costs span orders of\n    magnitude in 4D+ clouds), else unchanged.\'\'\'\n    v = np.asarray(values, dtype=float)\n    if COLOR_SCALE == \'log\':\n        return np.log10(np.clip(v, 1e-6, None))\n    return v\n\n\ndef color_scale_label(label):\n    return f\'log$_{{10}}$({label})\' if COLOR_SCALE == \'log\' else label\n\n\n# ---------------------------------------------------------------- markers\ndef _proj_2d(ax, xy):\n    \'\'\'Screen-space (data) coordinates of 2-D or projected 3-D points.\'\'\'\n    xy = np.asarray(xy, dtype=float)\n    if xy.shape[1] == 2:\n        return xy\n    from mpl_toolkits.mplot3d import proj3d\n    M = ax.get_proj()\n    return np.array([proj3d.proj_transform(p[0], p[1], p[2], M)[:2]\n                     for p in xy])\n\n\ndef number_points(ax, xy, indices=None, fontsize=12, offset=(9, 9),\n                  zorder=20, which=\'all\'):\n    \'\'\'Label measurements with their evaluation order.\n\n    `which`: \'all\' | \'key\' (first / best / last only) | \'none\'.\n    \'\'\'\n    if not SHOW_POINT_NUMBERS:\n        return\n    xy = np.asarray(xy, dtype=float)\n    idx = np.arange(len(xy)) if indices is None else np.asarray(indices)\n    try:\n        pts = _proj_2d(ax, xy)\n    except Exception:\n        return\n    if which == \'none\':\n        return\n    if which == \'key\':                   # a converged run piles 8 labels into\n        keep = {0, int(BEST_I), len(xy) - 1}      # one blob - keep 3 anchors\n        sel = [k for k, i in enumerate(idx) if int(i) in keep]\n        pts, idx = pts[sel], idx[sel]\n    for p, i in zip(pts, idx):\n        ax.annotate(str(int(i) + POINT_LABEL_BASE), xy=(p[0], p[1]),\n                    xycoords=\'data\', xytext=offset, textcoords=\'offset points\',\n                    fontsize=fontsize, fontweight=\'bold\', color=\'black\',\n                    zorder=zorder, annotation_clip=False,\n                    path_effects=[pe.withStroke(linewidth=2.6,\n                                                foreground=\'white\')])\n\n\ndef plot_measurements(ax, xy, sizes=(150, 150, 520), zorder=8, numbers=True):\n    \'\'\'Init / BO / flat / best measurements.\n\n    `xy` = (N, 2) or (N, 3) projected points, one row per evaluation.\n    `numbers` = True | False | \'all\' | \'key\' | \'none\' (see `number_points`).\n    \'\'\'\n    xy = np.asarray(xy, dtype=float)\n    is3d = xy.shape[1] == 3\n    idx = np.arange(len(xy))\n    ni = min(N_INIT, len(xy))\n    flat = IS_FLAT[:len(xy)]\n    m_init = (idx < ni) & ~flat\n    m_bo = (idx >= ni) & ~flat\n\n    def draw(mask, **kw):\n        if is3d:\n            kw[\'depthshade\'] = False\n        return ax.scatter(*[c[mask] for c in xy.T], **kw)\n\n    h = []\n    if m_init.any():\n        h.append(draw(m_init, s=sizes[0], marker=\'s\', c=C_INIT,\n                      edgecolors=\'black\', linewidths=1.4, zorder=zorder,\n                      label=f\'Initialization ({int(m_init.sum())})\'))\n    if m_bo.any():\n        h.append(draw(m_bo, s=sizes[1], marker=\'o\', c=C_BO,\n                      edgecolors=\'black\', linewidths=1.4, zorder=zorder,\n                      label=f\'BO iterations ({int(m_bo.sum())})\'))\n    if flat.any():\n        h.append(draw(flat, s=sizes[1] * 1.15, marker=\'v\', c=\'none\',\n                      edgecolors=C_FLAT, linewidths=2.6, zorder=zorder,\n                      label=f\'Flat spectrum, remapped ({int(flat.sum())})\'))\n    m_best = idx == BEST_I\n    h.append(draw(m_best, s=sizes[2], marker=\'*\', c=C_BEST,\n                  edgecolors=\'black\', linewidths=1.4, zorder=zorder + 1,\n                  label=f\'Best (cost = {COSTS_ALL[BEST_I]:.4g})\'))\n    if numbers:\n        number_points(ax, xy,\n                      which=numbers if isinstance(numbers, str) else \'all\')\n    return h\n\n\ndef plot_ghosts(ax, xy, size=110, zorder=7):\n    if not SHOW_GHOSTS or len(xy) == 0:\n        return None\n    xy = np.asarray(xy, dtype=float)\n    kw = dict(s=size, marker=\'x\', c=C_GHOST, linewidths=1.8, zorder=zorder,\n              label=f\'Virtual observations ({len(xy)})\')\n    return ax.scatter(*[c for c in xy.T], **kw)\n\n\ndef style_legend(ax, loc=\'upper right\', **kw):\n    leg = ax.legend(loc=loc, frameon=True, framealpha=0.94, edgecolor=\'0.3\',\n                    borderpad=0.6, labelspacing=0.5, handletextpad=0.5, **kw)\n    leg.get_frame().set_linewidth(1.2)\n    return leg\n\n\ndef figure_legend(fig, ax, y=-0.01, ncol=4, fontsize=17):\n    \'\'\'One horizontal legend under the whole figure (no overlap with panels).\'\'\'\n    handles, labels = ax.get_legend_handles_labels()\n    seen, h, l = set(), [], []\n    for hi, li in zip(handles, labels):\n        if li not in seen:\n            seen.add(li)\n            h.append(hi)\n            l.append(li)\n    leg = fig.legend(h, l, loc=\'lower center\', bbox_to_anchor=(0.5, y),\n                     ncol=ncol, frameon=True, framealpha=1.0, edgecolor=\'0.3\',\n                     borderpad=0.7, columnspacing=2.2, handletextpad=0.6,\n                     fontsize=fontsize)\n    leg.get_frame().set_linewidth(1.2)\n    return leg\n\n\nprint(\'plot helpers OK\')',
    "def _field_spec(field, projected=False):\n    '''(colormap, colorbar label, panel title) for 'mu' or 'sd'.'''\n    if field == 'mu':\n        return (CMAP_MU, COST_LABEL,\n                'Best achievable $\\\\mu$' if projected\n                else 'GP posterior mean $\\\\mu$')\n    return (CMAP_SD, 'GP standard deviation $\\\\sigma$',\n            '$\\\\sigma$ at the best point' if projected\n            else 'GP posterior standard deviation $\\\\sigma$')\n\n\ndef _add_top_axis(ax):\n    top = ax.secondary_xaxis('top',\n                             functions=(lambda v: QTOT_VAR - v,\n                                        lambda v: QTOT_VAR - v))\n    top.set_xlabel(f'{VAR_LABELS[1]} flow rate ({FLOW_UNIT})',\n                   labelpad=12)\n    top.tick_params(direction='in', labelsize=16, width=1.6, length=7)\n    return top\n\n\ndef fig_gp_1d(field='mu'):\n    x = np.linspace(0.0, QTOT_VAR, 600)\n    comp = np.column_stack([x, QTOT_VAR - x])\n    mu, sd = gp_eval(comp)\n    xb = COMP_REAL[BEST_I, 0]\n\n    fig, ax = plt.subplots(figsize=(FIG_SIZE, FIG_SIZE))\n    if field == 'mu':\n        ax.fill_between(x, mu - CI_FACTOR * sd, mu + CI_FACTOR * sd,\n                        color='#1f77b4', alpha=0.20, lw=0,\n                        label=f'GP $\\\\pm${CI_FACTOR:g}$\\\\sigma$')\n        ax.plot(x, mu, color='#10457a', lw=3.0, zorder=5,\n                label='GP posterior mean $\\\\mu$')\n        ax.axhline(0.0, color='0.6', lw=1.0, ls='--', zorder=1)\n        plot_measurements(ax, np.column_stack([COMP_REAL[:, 0], COSTS_PLOT]))\n        if N_GHOST:\n            gm, _ = gp_eval(COMP_GHOST)\n            plot_ghosts(ax, np.column_stack([COMP_GHOST[:, 0], gm]))\n        ax.set_ylabel(COST_LABEL)\n    else:\n        ax.plot(x, sd, color='#7b3294', lw=3.0, zorder=5,\n                label='GP posterior std. dev. $\\\\sigma$')\n        ax.fill_between(x, 0, sd, color='#7b3294', alpha=0.18, lw=0)\n        _, sd_at = gp_eval(COMP_REAL)\n        plot_measurements(ax, np.column_stack([COMP_REAL[:, 0], sd_at]))\n        ax.set_ylabel('GP standard deviation $\\\\sigma$')\n        ax.set_ylim(bottom=0)\n\n    ax.axvline(xb, color=C_BEST, lw=1.6, ls=':', zorder=2)\n    ax.set_xlim(0, QTOT_VAR)\n    ax.set_xlabel(f'{VAR_LABELS[0]} flow rate ({FLOW_UNIT})',\n                  labelpad=10)\n    ax.tick_params(direction='in', top=False, right=True)\n    style_legend(ax, loc='best', fontsize=14)\n    _add_top_axis(ax)\n    fig.suptitle(f'{EXPERIMENT_PATH.name} - GP surrogate, '\n                 f'$Q_{{tot}}$ = {QTOT_VAR:g} {FLOW_UNIT}', y=1.11)\n    return fig",
    "def _ternary_panel(ax, w, values, cmap, labels, levels=60, contour=True,\n                   title=None, flow_scale=None, axes_kw=None, ncontour=9,\n                   titlesize=None, titlepad=4):\n    xy = bary3(w)\n    tcf = ax.tricontourf(xy[:, 0], xy[:, 1], values, levels=levels, cmap=cmap,\n                         zorder=0)\n    for c in tcf.collections:\n        c.set_edgecolor('face')\n    if contour:\n        ax.tricontour(xy[:, 0], xy[:, 1], values, levels=ncontour,\n                      colors='white', linewidths=0.7, alpha=0.55, zorder=2)\n    draw_ternary_axes(ax, labels, flow_scale=flow_scale, **(axes_kw or {}))\n    if title:\n        ax.set_title(title, pad=titlepad,\n                     **({'fontsize': titlesize} if titlesize else {}))\n    return tcf\n\n\ndef ternary_legend(ax, ncol=1):\n    '''Compact legend anchored inside the panel, top-left of the triangle.'''\n    handles, labels = ax.get_legend_handles_labels()\n    seen, h, l = set(), [], []\n    for hi, li in zip(handles, labels):\n        if li not in seen:\n            seen.add(li)\n            h.append(hi)\n            l.append(li)\n    leg = ax.legend(h, l, loc='upper left', bbox_to_anchor=(-0.015, 0.97),\n                    ncol=ncol, frameon=True, framealpha=0.96, edgecolor='0.3',\n                    borderpad=0.5, labelspacing=0.4, handletextpad=0.45,\n                    columnspacing=1.1, fontsize=13.5)\n    leg.get_frame().set_linewidth(1.1)\n    return leg\n\n\ndef fig_gp_ternary(field='mu', keep=None):\n    keep = list(range(3)) if keep is None else list(keep)\n    w = simplex_lattice(RES_TERNARY, 3) / float(RES_TERNARY)\n    mu, sd = gp_eval(w * QTOT_VAR)\n    cmap, cbar, title = _field_spec(field)\n    values = mu if field == 'mu' else sd\n\n    fig, ax = plt.subplots(figsize=(FIG_SIZE, FIG_SIZE))\n    fig.subplots_adjust(top=0.90, bottom=0.02, left=0.02, right=0.98)\n    tcf = _ternary_panel(ax, w, values, cmap, [VAR_LABELS[i] for i in keep],\n                         contour=(field == 'mu'), flow_scale=QTOT_VAR)\n    plot_measurements(ax, bary3(normalise(COMP_REAL)), sizes=(130, 130, 480))\n    if N_GHOST:\n        plot_ghosts(ax, bary3(normalise(COMP_GHOST)))\n    add_colorbar(fig, tcf, ax, cbar, pad=0.0, shrink=0.72)\n    ax.set_aspect('equal', anchor='N')   # colorbar() resets the anchor - redo it\n    ternary_legend(ax)\n    fig.suptitle(f'{EXPERIMENT_PATH.name} - {title}  '\n                 f'($Q_{{tot}}$ = {QTOT_VAR:g} {FLOW_UNIT})', y=0.99)\n    return fig",
    'def _tetra_panel(ax, w, values, cmap, labels, alpha_mode=\'low\',\n                 title=None, size=9.0, gamma=3.0, hi=0.5, flow_scale=None,\n                 focus_q=None):\n    xyz = bary4(w)\n    cvals = to_color_scale(values)             # log10 if COLOR_SCALE == \'log\'\n    cm = plt.get_cmap(cmap)\n    if focus_q:                # keep only the informative part of the cloud:\n        keep = values <= np.quantile(values, float(focus_q))   # a 60k-point\n        xyz, values, cvals = xyz[keep], values[keep], cvals[keep]  # near-\n        size, hi, gamma = size * 2.6, min(1.0, hi * 1.5), 1.4      # transparent\n        alpha_mode = \'low\'     # haze has no shape; one dense blob does\n    # the colour scale spans whatever is actually drawn: stretched over the\n    # full field, a cropped cloud comes out uniformly one colour and its\n    # internal core-to-edge gradient - the reason to crop - is lost\n    norm = Normalize(vmin=float(np.min(cvals)), vmax=float(np.max(cvals)))\n    order = depth_sort(xyz, ax)\n    rgba = cm(norm(cvals[order]))\n    # opacity always keys off the RAW values, not the log-transformed ones -\n    # alpha_ramp\'s own dynamic-range handling already does the right thing\n    rgba[:, 3] = alpha_ramp(values[order], hi=hi, gamma=gamma, mode=alpha_mode)\n    ax.scatter(xyz[order, 0], xyz[order, 1], xyz[order, 2], s=size, c=rgba,\n               linewidths=0, marker=\'o\', depthshade=False, zorder=2)\n    draw_tetra_axes(ax, labels, flow_scale=flow_scale)\n    if title:\n        ax.set_title(title, y=0.94, fontsize=20)\n    return plt.cm.ScalarMappable(norm=norm, cmap=cm)\n\n\ndef _new_tetra_fig():\n    fig = plt.figure(figsize=(FIG_SIZE, FIG_SIZE))\n    ax = fig.add_subplot(111, projection=\'3d\')\n    ax.view_init(elev=18, azim=-58)\n    return fig, ax\n\n\ndef fig_gp_tetra(field=\'mu\', keep=None):\n    keep = list(range(4)) if keep is None else list(keep)\n    comp = sample_simplex(N_CLOUD)\n    mu, sd = gp_eval(comp)\n    w = normalise(comp[:, keep])\n    cmap, cbar, title = _field_spec(field)\n    values = mu if field == \'mu\' else sd\n\n    fig, ax = _new_tetra_fig()\n    sm = _tetra_panel(ax, w, values, cmap, [VAR_LABELS[i] for i in keep],\n                      alpha_mode=(TETRA_ALPHA_MODE if field == \'mu\' else \'high\'),\n                      flow_scale=QTOT_VAR,\n                      focus_q=(TETRA_CLOUD_Q if field == \'mu\' else None))\n    xyz_real = bary4(normalise(COMP_REAL[:, keep]))\n    if TETRA_SHADOWS:                    # before the markers: shadows sit under\n        plot_tetra_shadows(ax, xyz_real)\n    plot_measurements(ax, xyz_real, sizes=(150, 150, 520),\n                      numbers=TETRA_LABEL_POINTS)\n    if N_GHOST:\n        plot_ghosts(ax, bary4(normalise(COMP_GHOST[:, keep])), size=110)\n    add_colorbar(fig, sm, ax, color_scale_label(cbar) + (\n        f\' (best {TETRA_CLOUD_Q:.0%} shown)\'\n        if TETRA_CLOUD_Q and field == \'mu\' else \'\'),\n        pad=-0.04, shrink=0.58, fraction=0.030)\n    figure_legend(fig, ax, y=0.005, ncol=2)\n    fig.suptitle(f\'{EXPERIMENT_PATH.name} - {title}  \'\n                 f\'($Q_{{tot}}$ = {QTOT_VAR:g} {FLOW_UNIT})\', y=0.98)\n    # bottom margin has to clear the lowest vertex label ("PVP"), which the\n    # legend used to overprint\n    fig.subplots_adjust(left=0.02, right=0.94, bottom=0.17, top=0.94)\n    return fig',
    'PROJ_CACHE = {}\n\n\ndef proj_samples():\n    if \'comp\' not in PROJ_CACHE:\n        comp = sample_simplex(N_PROJ)\n        mu, sd = gp_eval(comp)\n        PROJ_CACHE.update(comp=comp, mu=mu, sd=sd)\n    return PROJ_CACHE[\'comp\'], PROJ_CACHE[\'mu\'], PROJ_CACHE[\'sd\']\n\n\ndef _proj_title(keep):\n    dropped = [VAR_NAMES[i] for i in range(N_VAR) if i not in keep]\n    return (f\'{EXPERIMENT_PATH.name} - best over \'\n            f\'{" and ".join(pretty(d) for d in dropped)}\')\n\n\ndef fig_proj_tetra(field=\'mu\', keep=None, nbins=None):\n    keep = KEEP4 if keep is None else list(keep)\n    nbins = PROJ_BINS if nbins is None else nbins\n    comp, mu, sd = proj_samples()\n    w, mp, sp = min_projection(comp, mu, sd, keep, nbins)\n    cmap, cbar, title = _field_spec(field, projected=True)\n    values = mp if field == \'mu\' else sp\n\n    fig, ax = _new_tetra_fig()\n    sm = _tetra_panel(ax, w, values, cmap, [VAR_LABELS[i] for i in keep],\n                      size=26, gamma=2.2, hi=0.85,\n                      alpha_mode=(TETRA_ALPHA_MODE if field == \'mu\' else \'high\'),\n                      flow_scale=QTOT_VAR,\n                      focus_q=(TETRA_CLOUD_Q if field == \'mu\' else None))\n    xyz_real = bary4(normalise(COMP_REAL[:, keep]))\n    if TETRA_SHADOWS:\n        plot_tetra_shadows(ax, xyz_real)\n    plot_measurements(ax, xyz_real, sizes=(150, 150, 520),\n                      numbers=TETRA_LABEL_POINTS)\n    add_colorbar(fig, sm, ax, color_scale_label(cbar), pad=-0.04, shrink=0.58,\n                fraction=0.030)\n    figure_legend(fig, ax, y=0.005, ncol=2)\n    fig.suptitle(f\'{_proj_title(keep)} - {title}\', y=0.98)\n    fig.subplots_adjust(left=0.02, right=0.94, bottom=0.17, top=0.94)\n    return fig\n\n\ndef fig_proj_ternary(field=\'mu\', keep=None, nbins=None):\n    keep = KEEP3 if keep is None else list(keep)\n    nbins = PROJ_BINS if nbins is None else nbins\n    comp, mu, sd = proj_samples()\n    w, mp, sp = min_projection(comp, mu, sd, keep, nbins)\n    cmap, cbar, title = _field_spec(field, projected=True)\n    values = mp if field == \'mu\' else sp\n\n    fig, ax = plt.subplots(figsize=(FIG_SIZE, FIG_SIZE))\n    fig.subplots_adjust(top=0.90, bottom=0.02, left=0.02, right=0.98)\n    tcf = _ternary_panel(ax, w, values, cmap, [VAR_LABELS[i] for i in keep],\n                         contour=(field == \'mu\'), flow_scale=QTOT_VAR)\n    plot_measurements(ax, bary3(normalise(COMP_REAL[:, keep])),\n                      sizes=(130, 130, 480))\n    add_colorbar(fig, tcf, ax, cbar, pad=0.0, shrink=0.72)\n    ax.set_aspect(\'equal\', anchor=\'N\')   # colorbar() resets the anchor - redo it\n    ternary_legend(ax)\n    fig.suptitle(f\'{_proj_title(keep)} - {title}\', y=0.99)\n    return fig',
    "def _isopleth_keep(hold):\n    '''The 3 reagents on the triangle, plus any surplus to min-project.'''\n    rank = list(np.argsort(-axis_influence()))\n    keep = sorted([i for i in rank if i != hold][:3])\n    extra = [i for i in range(N_VAR) if i != hold and i not in keep]\n    return keep, extra\n\n\ndef isopleth_field(c, hold, keep, extra, res=None, n=300000, nbins=32):\n    '''GP field on the ternary section {x_hold = c}, renormalised over `keep`.\n\n    No `extra` -> exact lattice evaluation of the section. Otherwise the\n    surplus reagents are collapsed by `min_projection`, i.e. every cell keeps\n    the best cost reachable at that (kept, held) composition.\n    '''\n    res = ISOPLETH_RES if res is None else res\n    held = c * QTOT_VAR\n    budget = (1.0 - c) * QTOT_VAR\n    if not extra:\n        lat = simplex_lattice(res, 3) / float(res)\n        comp = np.zeros((len(lat), N_VAR))\n        comp[:, keep] = lat * budget\n        comp[:, hold] = held\n        mu, sd = gp_eval(comp)\n        return lat, mu, sd\n    # a min-projection is only as smooth as the number of samples that land\n    # in each cell, so the section uses more samples and coarser bins than the\n    # full-simplex projections - otherwise the panels speckle\n    free = list(keep) + list(extra)\n    comp = np.zeros((n, N_VAR))\n    comp[:, free] = RNG.dirichlet(np.ones(len(free)), size=n) * budget\n    comp[:, hold] = held\n    mu, sd = gp_eval(comp)\n    return min_projection(comp, mu, sd, keep, nbins)\n\n\ndef _isopleth_measurements(ax, hold, keep, c, tol=None):\n    '''Measurements lying in the slab |x_hold - c| < tol, at their\n    renormalised position. Out-of-slab rows are blanked with NaN rather than\n    dropped, so evaluation indices - and the shared legend counts - stay those\n    of the whole run.'''\n    tol = ISOPLETH_TOL if tol is None else tol\n    frac = COMP_REAL[:, hold] / QTOT_VAR\n    xy = bary3(normalise(COMP_REAL[:, keep])).copy()\n    xy[np.abs(frac - c) > tol] = np.nan\n    return plot_measurements(ax, xy, sizes=(80, 80, 300), numbers='none')\n\n\ndef fig_gp_isopleths(field='mu', hold=None, levels=None, res=None):\n    if hold is None:\n        hold = (int(np.argmax(axis_influence())) if ISOPLETH_HOLD is None\n                else int(ISOPLETH_HOLD))\n    hold = int(hold)\n    levels = ISOPLETH_LEVELS if levels is None else levels\n    keep, extra = _isopleth_keep(hold)\n    cmap, cbar, title = _field_spec(field, projected=bool(extra))\n\n    fields = []\n    for c in levels:\n        w, mu, sd = isopleth_field(c, hold, keep, extra, res=res)\n        fields.append((c, w, mu if field == 'mu' else sd))\n    vmin = min(float(np.min(v)) for _, _, v in fields)\n    vmax = max(float(np.max(v)) for _, _, v in fields)\n    lv = np.linspace(vmin, vmax, 60)           # shared -> panels comparable\n\n    # Explicit inch layout. `subplots_adjust` cannot be trusted here: the\n    # panels have a fixed aspect, so any fraction-based margin either\n    # letterboxes the triangles or lets the titles, the corner labels and the\n    # legend collide - all four happen at once when the strip is short.\n    ratio = 1.40 / (_SQ3 + 0.18)         # the xlim/ylim spans of a ternary ax\n    n = len(fields)\n    wp = float(ISOPLETH_PANEL)\n    hp, gap = wp / ratio, 0.42\n    ml, mr, mt, mb = 0.12, 1.15, 1.36, 0.92    # colorbar / titles / legend\n    fw = ml + n * wp + (n - 1) * gap + mr\n    fh = mt + hp + mb\n    fig = plt.figure(figsize=(fw, fh))\n    axes = np.array([fig.add_axes([(ml + k * (wp + gap)) / fw, mb / fh,\n                                   wp / fw, hp / fh]) for k in range(n)])\n    axes_kw = dict(fontsize=11.5, ticksize=9, lw=1.4, label_pad=0.075,\n                   nticks=4)\n    for ax, (c, w, values) in zip(axes, fields):\n        tcf = _ternary_panel(ax, w, values, cmap,\n                             [VAR_LABELS[i] for i in keep], levels=lv,\n                             contour=(field == 'mu'), ncontour=7,\n                             flow_scale=(1.0 - c) * QTOT_VAR,\n                             axes_kw=axes_kw,\n                             # two lines and a generous pad: a one-line title\n                             # is wider than the panel and lands on both the\n                             # apex label and the neighbouring triangle\n                             titlesize=12.5, titlepad=16,\n                             title=f'{VAR_LABELS[hold]} = {c:.0%}\\n'\n                                   f'({c * QTOT_VAR:.2f} {FLOW_UNIT})')\n        _isopleth_measurements(ax, hold, keep, c)\n        ax.set_aspect('equal', anchor='N')\n    cax = fig.add_axes([(fw - mr + 0.30) / fw, (mb + 0.10 * hp) / fh,\n                        0.16 / fw, 0.78 * hp / fh])\n    cb = fig.colorbar(tcf, cax=cax)\n    cb.locator = mpl.ticker.MaxNLocator(5)     # 60 contour levels would\n    cb.update_ticks()                          # otherwise all become ticks\n    cb.set_label(color_scale_label(cbar) if field == 'mu' else cbar,\n                 fontsize=12.5, labelpad=8)\n    cb.ax.tick_params(labelsize=10, width=1.0, length=4)\n    cb.outline.set_linewidth(1.2)\n    figure_legend(fig, axes[0], y=0.10 / fh, ncol=5, fontsize=12)\n    fig.suptitle(f'{EXPERIMENT_PATH.name} - {title}, sections at fixed '\n                 f'{pretty(VAR_NAMES[hold])}'\n                 f'  ($Q_{{tot}}$ = {QTOT_VAR:g} {FLOW_UNIT})',\n                 y=1.0 - 0.20 / fh, fontsize=14, va='top')\n    return fig",
    "CORNER_BINS = 64         # grid per axis on each pair map\nCORNER_LATTICE_2 = 28    # subdivisions of the collapsed directions when two\n                         # reagents are collapsed (4 pumps)\nCORNER_LATTICE_3 = 14    # ... when three or more are (5 pumps: the lattice\n                         # grows as L^2, so this is what keeps it under a\n                         # minute)\nCORNER_LATTICE_1 = 40    # ... on the diagonal profiles\nCORNER_PANEL = 2.6       # inches per panel\nCORNER_LABEL_POINTS = 'all'   # 'all' | 'key' | 'none' - evaluation numbers\n\n\ndef _rest_lattice(m, budget, L):\n    '''Regular lattice of the (m-1)-simplex scaled to `budget`.'''\n    if m <= 0:\n        return np.zeros((1, 0))\n    if m == 1:\n        return np.full((1, 1), budget)\n    return simplex_lattice(L, m) / float(L) * budget\n\n\ndef min_pair_grid(i, j, nbins=None, L=None):\n    '''Best mu (and the sigma of that winning point) on a grid of the\n    (flow_i, flow_j) triangle, minimising exactly over the other reagents.'''\n    nbins = CORNER_BINS if nbins is None else nbins\n    rest = [k for k in range(N_VAR) if k not in (i, j)]\n    if L is None:\n        L = CORNER_LATTICE_2 if len(rest) <= 2 else CORNER_LATTICE_3\n    edges = np.linspace(0.0, QTOT_VAR, nbins + 1)\n    cen = 0.5 * (edges[:-1] + edges[1:])\n    blocks, meta = [], []\n    for a, xi in enumerate(cen):\n        for b, xj in enumerate(cen):\n            r = QTOT_VAR - xi - xj\n            if r <= 1e-9:\n                continue\n            sub = _rest_lattice(len(rest), r, L)\n            comp = np.zeros((len(sub), N_VAR))\n            comp[:, i] = xi\n            comp[:, j] = xj\n            if rest:\n                comp[:, rest] = sub\n            blocks.append(comp)\n            meta.append((b, a, len(sub)))\n    mu, sd = gp_eval(np.vstack(blocks))\n    M = np.full((nbins, nbins), np.nan)     # rows = j (y), cols = i (x)\n    S = np.full((nbins, nbins), np.nan)\n    off = 0\n    for b, a, k in meta:\n        seg = mu[off:off + k]\n        w = int(np.argmin(seg))\n        M[b, a], S[b, a] = seg[w], sd[off + w]\n        off += k\n    return cen, M, S\n\n\ndef min_axis_curve(i, nbins=None, L=None):\n    '''Best mu reachable at each flow rate of reagent `i`, plus the sigma of\n    the winning point (the diagonal of the matrix).'''\n    nbins = CORNER_BINS if nbins is None else nbins\n    L = CORNER_LATTICE_1 if L is None else L\n    rest = [k for k in range(N_VAR) if k != i]\n    edges = np.linspace(0.0, QTOT_VAR, nbins + 1)\n    cen = 0.5 * (edges[:-1] + edges[1:])\n    blocks, sizes = [], []\n    for xi in cen:\n        sub = _rest_lattice(len(rest), max(QTOT_VAR - xi, 0.0), L)\n        comp = np.zeros((len(sub), N_VAR))\n        comp[:, i] = xi\n        comp[:, rest] = sub\n        blocks.append(comp)\n        sizes.append(len(sub))\n    mu, sd = gp_eval(np.vstack(blocks))\n    best_mu, best_sd, off = [], [], 0\n    for k in sizes:\n        seg = mu[off:off + k]\n        w = int(np.argmin(seg))\n        best_mu.append(seg[w])\n        best_sd.append(sd[off + w])\n        off += k\n    return cen, np.array(best_mu), np.array(best_sd)\n\n\ndef fig_gp_corner(field='mu', nbins=None):\n    n = N_VAR\n    pairs = {(r, c): min_pair_grid(c, r, nbins=nbins)\n             for r in range(n) for c in range(r)}\n    diag = {r: min_axis_curve(r, nbins=nbins) for r in range(n)}\n    cmap, cbar, _ = _field_spec(field, projected=True)\n    grids = {k: (v[1] if field == 'mu' else v[2]) for k, v in pairs.items()}\n    vmin = min(np.nanmin(g) for g in grids.values())\n    vmax = max(np.nanmax(g) for g in grids.values())\n    norm = Normalize(vmin=float(vmin), vmax=float(vmax))\n    # the diagonals share a y range set by the CURVES, not by the sigma band:\n    # the band is several times the signal here, and letting it drive the\n    # limits flattens every profile into a straight line\n    cy = np.concatenate([diag[r][1] for r in range(n)])\n    pad = 0.10 * ((cy.max() - cy.min()) or 1.0)\n    ylim = (cy.min() - pad, cy.max() + pad)\n\n    wp = float(CORNER_PANEL)\n    ml, mr, mt, mb, gap = 1.00, 2.05, 0.62, 1.55, 0.14\n    fw = ml + n * wp + (n - 1) * gap + mr\n    fh = mt + n * wp + (n - 1) * gap + mb\n    fig = plt.figure(figsize=(fw, fh))\n\n    def cell(r, c):\n        return fig.add_axes([(ml + c * (wp + gap)) / fw,\n                             (mb + (n - 1 - r) * (wp + gap)) / fh,\n                             wp / fw, wp / fh])\n\n    tick = dict(labelsize=11.5, width=1.2, length=4.5, direction='in')\n    im, leg_ax = None, None\n    for r in range(n):\n        for c in range(r + 1):\n            ax = cell(r, c)\n            ax.tick_params(**tick)\n            if c < r:\n                cen, M, S = pairs[(r, c)]\n                im = ax.imshow(M if field == 'mu' else S, origin='lower',\n                               extent=(0, QTOT_VAR, 0, QTOT_VAR), cmap=cmap,\n                               norm=norm, interpolation='nearest',\n                               aspect='equal', zorder=0)\n                ax.plot([0, QTOT_VAR], [QTOT_VAR, 0], color='0.35', lw=1.0,\n                        zorder=3)     # the no-other-reagent boundary\n                xy = np.column_stack([COMP_REAL[:, c], COMP_REAL[:, r]])\n                plot_measurements(ax, xy, sizes=(46, 46, 210), numbers=False)\n                number_points(ax, xy, fontsize=10, offset=(5, 4),\n                              which=CORNER_LABEL_POINTS)\n                ax.set_xlim(0, QTOT_VAR)\n                ax.set_ylim(0, QTOT_VAR)\n                leg_ax = ax\n            else:                      # diagonal: the 1-D reduction\n                x, m, sdv = diag[r]\n                ax.fill_between(x, m - CI_FACTOR * sdv, m + CI_FACTOR * sdv,\n                                color=C_BO, alpha=0.18, lw=0)\n                ax.plot(x, m, color=C_BO, lw=2.0)\n                ax.axvline(COMP_REAL[BEST_I, r], color=C_BEST, lw=1.4,\n                           ls='--', alpha=0.85)\n                ax.set_xlim(0, QTOT_VAR)\n                ax.set_ylim(*ylim)\n                ax.set_facecolor('0.97')\n                ax.yaxis.set_label_position('right')\n                ax.yaxis.tick_right()\n                if r == n - 1:       # once, not on every diagonal panel\n                    ax.set_ylabel('best ' + ('$\\\\mu$' if field == 'mu'\n                                             else '$\\\\sigma$'),\n                                  fontsize=13, labelpad=10)\n                ax.text(0.5, 0.95, VAR_LABELS[r], transform=ax.transAxes,\n                        ha='center', va='top', fontsize=13.5,\n                        fontweight='bold')\n            if c == 0 and r > 0:\n                ax.set_ylabel(f'{VAR_LABELS[r]}\\n({FLOW_UNIT})', fontsize=13.5)\n            elif c > 0 and c < r:\n                ax.set_yticklabels([])\n            if r == n - 1:\n                ax.set_xlabel(f'{VAR_LABELS[c]}\\n({FLOW_UNIT})', fontsize=13.5)\n            else:\n                ax.set_xticklabels([])\n\n    cax = fig.add_axes([(fw - mr + 1.10) / fw, mb / fh, 0.18 / fw,\n                        (n * wp + (n - 1) * gap) * 0.5 / fh])\n    cb = fig.colorbar(im, cax=cax)\n    cb.set_label(color_scale_label(cbar) if field == 'mu' else cbar,\n                 fontsize=14, labelpad=10)\n    cb.ax.tick_params(labelsize=11.5, width=1.2, length=5)\n    cb.outline.set_linewidth(1.3)\n    if leg_ax is not None:\n        figure_legend(fig, leg_ax, y=0.12 / fh, ncol=4, fontsize=12.5)\n    sym = '$\\\\mu$' if field == 'mu' else '$\\\\sigma$'\n    fig.suptitle(f'{EXPERIMENT_PATH.name} - best achievable {sym} per reagent '\n                 f'pair  ($Q_{{tot}}$ = {QTOT_VAR:g} {FLOW_UNIT}, '\n                 f'star = best evaluation)',\n                 y=1.0 - 0.16 / fh, fontsize=15, va='top')\n    return fig",
    "def fig_bestslice_triptych(keep=None, nbins=None):\n    keep = KEEP3 if keep is None else list(keep)\n    nbins = PROJ_BINS if nbins is None else nbins\n    comp, mu, sd = proj_samples()\n    w, mp, sp, dv = min_projection(comp, mu, sd, keep, nbins,\n                                   return_argmin=True)\n    drop = [i for i in range(N_VAR) if i not in keep]\n    labels = [VAR_LABELS[i] for i in keep]\n\n    panels = [(mp, CMAP_MU, color_scale_label(COST_LABEL), True,\n               'Best achievable $\\\\mu$')]\n    for k, d in enumerate(drop):\n        panels.append((dv[:, k], 'cividis',\n                       f'{VAR_LABELS[d]} at that point ({FLOW_UNIT})', False,\n                       f'{pretty(VAR_NAMES[d])} of the winning point'))\n    panels.append((sp, CMAP_SD, 'GP standard deviation $\\\\sigma$', False,\n                   '$\\\\sigma$ at that point'))\n\n    ratio = 1.40 / (_SQ3 + 0.18)\n    n = len(panels)\n    wp = 3.35\n    hp, gap = wp / ratio, 0.34\n    ml, mr, mt, mb = 0.12, 0.12, 1.05, 1.80     # colorbars + legend below\n    fw = ml + n * wp + (n - 1) * gap + mr\n    fh = mt + hp + mb\n    fig = plt.figure(figsize=(fw, fh))\n    axes_kw = dict(fontsize=12, ticksize=9.5, lw=1.5, label_pad=0.075,\n                   nticks=4)\n    for k, (values, cmap, cbl, contour, title) in enumerate(panels):\n        left = (ml + k * (wp + gap)) / fw\n        ax = fig.add_axes([left, mb / fh, wp / fw, hp / fh])\n        tcf = _ternary_panel(ax, w, values, cmap, labels, contour=contour,\n                             ncontour=7, flow_scale=QTOT_VAR,\n                             axes_kw=axes_kw, titlesize=13, titlepad=10,\n                             title=title)\n        plot_measurements(ax, bary3(normalise(COMP_REAL[:, keep])),\n                          sizes=(85, 85, 320), numbers='none')\n        ax.set_aspect('equal', anchor='N')\n        cax = fig.add_axes([left + 0.10 * wp / fw, 1.02 / fh,\n                            0.80 * wp / fw, 0.15 / fh])\n        cb = fig.colorbar(tcf, cax=cax, orientation='horizontal')\n        cb.locator = mpl.ticker.MaxNLocator(5)\n        cb.update_ticks()\n        cb.set_label(cbl, fontsize=11, labelpad=4)\n        cb.ax.tick_params(labelsize=9, width=1.0, length=4)\n        cb.outline.set_linewidth(1.1)\n        if k == 0:\n            leg_ax = ax\n    figure_legend(fig, leg_ax, y=0.06 / fh, ncol=4, fontsize=11)\n    fig.suptitle(f'{_proj_title(keep)}  ($Q_{{tot}}$ = {QTOT_VAR:g} '\n                 f'{FLOW_UNIT})', y=1.0 - 0.18 / fh, fontsize=14, va='top')\n    return fig",
    "def make_static_figures():\n    made = {}\n    if N_VAR < 2:\n        print('less than 2 varying pumps: nothing to map')\n        return made\n    for field, tag in (('mu', 'mean'), ('sd', 'std')):\n        if N_VAR == 2:\n            made[f'gp_{tag}_2pumps'] = fig_gp_1d(field)\n        elif N_VAR == 3:\n            made[f'gp_{tag}_ternary_3pumps'] = fig_gp_ternary(field)\n        elif N_VAR == 4:\n            made[f'gp_{tag}_tetrahedron_4pumps'] = fig_gp_tetra(field)\n            made[f'gp_{tag}_bestslice_ternary'] = fig_proj_ternary(field)\n            made[f'gp_{tag}_isopleths'] = fig_gp_isopleths(field)\n            made[f'gp_{tag}_corner'] = fig_gp_corner(field)\n        else:\n            made[f'gp_{tag}_projection_tetrahedron'] = fig_proj_tetra(field)\n            made[f'gp_{tag}_projection_ternary'] = fig_proj_ternary(field)\n            made[f'gp_{tag}_isopleths'] = fig_gp_isopleths(field)\n            made[f'gp_{tag}_corner'] = fig_gp_corner(field)\n    if N_VAR >= 4:                       # one figure, not one per field\n        made['gp_bestslice_triptych'] = fig_bestslice_triptych()\n    if N_VAR == 4:\n        print(f'4 varying pumps -> best-slice ternary drops '\n              f'{[VAR_NAMES[i] for i in range(N_VAR) if i not in KEEP3][0]}')\n    elif N_VAR >= 5:\n        print(f'{N_VAR} varying pumps -> min-projections '\n              f'(tetra keeps {[VAR_NAMES[i] for i in KEEP4]}, '\n              f'ternary keeps {[VAR_NAMES[i] for i in KEEP3]})')\n    for name, fig in made.items():\n        save_fig(fig, name)\n    return made\n\n\nSTATIC_FIGS = make_static_figures()\nplt.show()",
    'PLOTLY_FONT = dict(family=\'Arial, Helvetica, sans-serif\', size=18,\n                   color=\'black\')\n\n\ndef _axis_title(i):\n    return f\'Q({VAR_NAMES[i]})\'\n\n\ndef _plotly_layout(fig, title, scene_axes, height=820, subtitle=None):\n    text = title\n    if subtitle:                       # keep the long qualifiers off the main\n        text += (\'<br><span style="font-size:15px;color:#555">\'   # title line,\n                 f\'{subtitle}</span>\')                            # which clips\n    fig.update_layout(\n        title=dict(text=text, x=0.5, xanchor=\'center\',\n                   font=dict(size=22, family=PLOTLY_FONT[\'family\'])),\n        font=PLOTLY_FONT, height=height, template=\'plotly_white\',\n        margin=dict(l=10, r=10, t=(110 if subtitle else 80), b=10),\n        scene=dict(\n            xaxis=dict(title=dict(text=scene_axes[0], font=dict(size=19)),\n                       tickfont=dict(size=15)),\n            yaxis=dict(title=dict(text=scene_axes[1], font=dict(size=19)),\n                       tickfont=dict(size=15)),\n            zaxis=dict(title=dict(text=scene_axes[2], font=dict(size=19)),\n                       tickfont=dict(size=15)),\n            aspectmode=\'cube\',\n            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))),\n    )\n    return fig\n\n\ndef _measure_traces(coords3, ghost3=None):\n    \'\'\'Scatter3d traces for init / BO / best, given (N,3) coordinates.\n\n    `ghost3` must be the ghost points expressed in the SAME 3 axes as\n    `coords3` - they are not always the first three composition columns\n    (the 5-pump slider plots free = others[:3], and the 3-pump surface uses\n    cost as its z axis).\n    \'\'\'\n    coords3 = np.asarray(coords3, dtype=float)\n    ni = min(N_INIT, len(coords3))\n    txt = [f\'#{i} cost={COSTS_ALL[i]:.4g}<br>\'\n           + \'<br>\'.join(f\'{n}={COMP_REAL[i, k]:.3f}\'\n                         for k, n in enumerate(VAR_NAMES))\n           for i in range(len(coords3))]\n    tr = []\n    if ni:\n        tr.append(go.Scatter3d(\n            x=coords3[:ni, 0], y=coords3[:ni, 1], z=coords3[:ni, 2],\n            mode=\'markers\', name=f\'Initialization ({ni})\', text=txt[:ni],\n            hoverinfo=\'text\',\n            marker=dict(size=7, color=C_INIT, symbol=\'square\',\n                        line=dict(color=\'black\', width=2))))\n    if len(coords3) > ni:\n        tr.append(go.Scatter3d(\n            x=coords3[ni:, 0], y=coords3[ni:, 1], z=coords3[ni:, 2],\n            mode=\'markers\', name=f\'BO iterations ({len(coords3) - ni})\',\n            text=txt[ni:], hoverinfo=\'text\',\n            marker=dict(size=7, color=C_BO,\n                        line=dict(color=\'black\', width=2))))\n    b = coords3[BEST_I]\n    tr.append(go.Scatter3d(\n        x=[b[0]], y=[b[1]], z=[b[2]], mode=\'markers\',\n        name=f\'Best (cost = {COSTS_ALL[BEST_I]:.4g})\',\n        text=[txt[BEST_I]], hoverinfo=\'text\',\n        marker=dict(size=14, color=C_BEST, symbol=\'diamond\',\n                    line=dict(color=\'black\', width=2))))\n    if SHOW_GHOSTS and N_GHOST and ghost3 is not None:\n        g = np.asarray(ghost3, dtype=float)\n        tr.append(go.Scatter3d(\n            x=g[:, 0], y=g[:, 1], z=g[:, 2],\n            mode=\'markers\', name=f\'Virtual observations ({len(g)})\',\n            marker=dict(size=5, color=C_GHOST, symbol=\'x\', opacity=0.75)))\n    return tr\n\n\ndef plotly_surface_3pumps():\n    \'\'\'3 varying pumps: mu as a surface over (q1, q2), q3 = Qtot - q1 - q2.\'\'\'\n    q = np.linspace(0, QTOT_VAR, RES_SURF3D)\n    Q1, Q2 = np.meshgrid(q, q)\n    Q3 = QTOT_VAR - Q1 - Q2\n    ok = Q3 >= 0\n    comp = np.column_stack([Q1[ok], Q2[ok], Q3[ok]])\n    mu, sd = gp_eval(comp)\n    Zm = np.full(Q1.shape, np.nan); Zm[ok] = mu\n    Zl = np.full(Q1.shape, np.nan); Zl[ok] = mu - sd\n    Zh = np.full(Q1.shape, np.nan); Zh[ok] = mu + sd\n\n    fig = go.Figure()\n    fig.add_trace(go.Surface(x=Q1, y=Q2, z=Zh, showscale=False, opacity=0.22,\n                             colorscale=\'Greys\', name=\'mu + sigma\',\n                             hoverinfo=\'skip\'))\n    fig.add_trace(go.Surface(x=Q1, y=Q2, z=Zl, showscale=False, opacity=0.22,\n                             colorscale=\'Greys\', name=\'mu - sigma\',\n                             hoverinfo=\'skip\'))\n    fig.add_trace(go.Surface(\n        x=Q1, y=Q2, z=Zm, colorscale=\'Viridis_r\', name=\'GP mean\',\n        colorbar=dict(title=dict(text=COST_LABEL, side=\'right\',\n                                 font=dict(size=19)),\n                      tickfont=dict(size=15), len=0.72),\n        contours=dict(z=dict(show=True, usecolormap=True, project_z=True,\n                             width=2))))\n    ghost3 = None\n    if N_GHOST:                       # z axis is cost here, so ghosts sit at\n        gm, _ = gp_eval(COMP_GHOST)   # their GP value, not at q3\n        ghost3 = np.column_stack([COMP_GHOST[:, 0], COMP_GHOST[:, 1], gm])\n    for t in _measure_traces(np.column_stack([COMP_REAL[:, 0], COMP_REAL[:, 1],\n                                              COSTS_PLOT]), ghost3):\n        fig.add_trace(t)\n    return _plotly_layout(\n        fig, f\'{EXPERIMENT_PATH.name} - GP surrogate \'\n             f\'(Qtot = {QTOT_VAR:g})\',\n        [_axis_title(0), _axis_title(1), COST_LABEL])\n\n\ndef _volume_grid(res, free_idx, fixed=None, total=None):\n    \'\'\'Regular grid over `free_idx` with the remaining flows derived.\'\'\'\n    total = QTOT_VAR if total is None else total\n    g = np.linspace(0, total, res)\n    G = np.meshgrid(g, g, g, indexing=\'ij\')\n    comp = np.zeros((G[0].size, N_VAR))\n    for k, i in enumerate(free_idx):\n        comp[:, i] = G[k].ravel()\n    if fixed:\n        for i, v in fixed.items():\n            comp[:, i] = v\n    rest = [i for i in range(N_VAR) if i not in free_idx and\n            (not fixed or i not in fixed)]\n    if rest:\n        comp[:, rest[0]] = QTOT_VAR - comp.sum(axis=1)\n    ok = feasible(comp)\n    return comp, ok, [G[k].ravel() for k in range(3)]\n\n\ndef focus_range(vals, focus):\n    \'\'\'Isosurface range that keeps the informative end of the field.\'\'\'\n    v = np.asarray(vals, dtype=float)\n    v = v[np.isfinite(v)]\n    q = float(np.clip(VOLUME_FOCUS_Q, 0.05, 1.0))\n    if focus == \'low\':                      # cost: show the good (low) end\n        return float(v.min()), float(np.quantile(v, q))\n    return float(np.quantile(v, 1.0 - q)), float(v.max())\n\n\ndef _volume_trace(xyz, values, ok, cmap, label, visible=True, cmin=None,\n                  cmax=None, focus=\'low\'):\n    # go.Volume silently drops the whole grid if any value is NaN, so points\n    # outside the simplex get a sentinel far above isomax instead of a NaN.\n    if cmin is None or cmax is None:\n        lo, hi = focus_range(values[ok], focus)\n    else:\n        lo, hi = float(cmin), float(cmax)\n    span = max(hi - lo, 1e-9)\n    v = np.where(ok & np.isfinite(values), values, hi + 1e3 * span)\n    return go.Volume(\n        x=xyz[0], y=xyz[1], z=xyz[2], value=v,\n        isomin=lo, isomax=hi, cmin=lo, cmax=hi,\n        opacity=0.12, surface_count=19,\n        colorscale=cmap, visible=visible, name=label,\n        colorbar=dict(title=dict(text=label, side=\'right\',\n                                 font=dict(size=19)),\n                      tickfont=dict(size=15), len=0.72),\n        caps=dict(x_show=False, y_show=False, z_show=False))\n\n\ndef plotly_volume_4pumps():\n    \'\'\'4 varying pumps: volumetric rendering of the 3-simplex.\'\'\'\n    free = [0, 1, 2]\n    comp, ok, xyz = _volume_grid(RES_VOLUME, free)\n    mu = np.full(len(comp), np.nan)\n    sd = np.full(len(comp), np.nan)\n    mu[ok], sd[ok] = gp_eval(comp[ok])\n\n    fig = go.Figure()\n    fig.add_trace(_volume_trace(xyz, mu, ok, \'Viridis_r\', COST_LABEL, True,\n                                focus=\'low\'))\n    fig.add_trace(_volume_trace(xyz, sd, ok, \'Magma\',\n                                \'GP standard deviation\', False,\n                                focus=\'high\'))\n    n_vol = 2\n    for t in _measure_traces(COMP_REAL[:, free]):\n        fig.add_trace(t)\n    n_pts = len(fig.data) - n_vol\n\n    def vis(k):\n        return [i == k for i in range(n_vol)] + [True] * n_pts\n\n    fig.update_layout(updatemenus=[dict(\n        type=\'buttons\', direction=\'right\', x=0.5, xanchor=\'center\',\n        y=1.06, yanchor=\'bottom\', showactive=True, font=dict(size=17),\n        buttons=[dict(label=\'posterior mean\', method=\'update\',\n                      args=[{\'visible\': vis(0)}]),\n                 dict(label=\'posterior std\', method=\'update\',\n                      args=[{\'visible\': vis(1)}])])])\n    return _plotly_layout(\n        fig, f\'{EXPERIMENT_PATH.name} - GP surrogate over the quaternary \'\n             f\'simplex (Q({VAR_NAMES[3]}) = Qtot - sum)\',\n        [_axis_title(i) for i in free])\n\n\ndef plotly_volume_5pumps(slice_dim=None):\n    \'\'\'5 varying pumps: volume over 3 reagents, slider on one of the others.\'\'\'\n    d = SLICE_DIM if slice_dim is None else int(slice_dim)\n    others = [i for i in range(N_VAR) if i != d]\n    free, derived = others[:3], others[3]\n    slices = np.linspace(0, QTOT_VAR * 0.75, N_SLICES_5D)\n\n    frames, first = [], None\n    lo_mu = hi_mu = lo_sd = hi_sd = None\n    packed = []\n    for s in slices:\n        g = np.linspace(0, QTOT_VAR - s, RES_VOLUME_5D)\n        G = np.meshgrid(g, g, g, indexing=\'ij\')\n        comp = np.zeros((G[0].size, N_VAR))\n        for k, i in enumerate(free):\n            comp[:, i] = G[k].ravel()\n        comp[:, d] = s\n        comp[:, derived] = QTOT_VAR - comp.sum(axis=1)\n        ok = feasible(comp)\n        mu = np.full(len(comp), np.nan)\n        sd = np.full(len(comp), np.nan)\n        if ok.any():\n            mu[ok], sd[ok] = gp_eval(comp[ok])\n        packed.append(([G[k].ravel() for k in range(3)], mu, sd, ok, s))\n    allmu = np.concatenate([p[1][p[3]] for p in packed if p[3].any()])\n    allsd = np.concatenate([p[2][p[3]] for p in packed if p[3].any()])\n    lo_mu, hi_mu = focus_range(allmu, \'low\')\n    lo_sd, hi_sd = focus_range(allsd, \'high\')\n\n    for xyz, mu, sd, ok, s in packed:\n        tr = [_volume_trace(xyz, mu, ok, \'Viridis_r\', COST_LABEL, True,\n                            lo_mu, hi_mu),\n              _volume_trace(xyz, sd, ok, \'Magma\', \'GP standard deviation\',\n                            False, lo_sd, hi_sd)]\n        frames.append(go.Frame(data=tr, name=f\'{s:.2f}\'))\n        if first is None:\n            first = tr\n\n    fig = go.Figure(data=first, frames=frames)\n    n_vol = 2\n    for t in _measure_traces(COMP_REAL[:, free]):\n        fig.add_trace(t)\n    n_pts = len(fig.data) - n_vol\n\n    def vis(k):\n        return [i == k for i in range(n_vol)] + [True] * n_pts\n\n    fig.update_layout(\n        sliders=[dict(\n            active=0, x=0.08, len=0.84, y=0.02, pad=dict(t=40),\n            currentvalue=dict(prefix=f\'Q({VAR_NAMES[d]}) = \', suffix=\' \',\n                              font=dict(size=20)),\n            font=dict(size=15),\n            steps=[dict(method=\'animate\', label=f\'{s:.2f}\',\n                        args=[[f\'{s:.2f}\'],\n                              dict(mode=\'immediate\',\n                                   frame=dict(duration=0, redraw=True),\n                                   transition=dict(duration=0))])\n                   for s in slices])],\n        updatemenus=[dict(\n            type=\'buttons\', direction=\'right\', x=0.5, xanchor=\'center\',\n            y=1.06, yanchor=\'bottom\', showactive=True, font=dict(size=17),\n            buttons=[dict(label=\'posterior mean\', method=\'update\',\n                          args=[{\'visible\': vis(0)}]),\n                     dict(label=\'posterior std\', method=\'update\',\n                          args=[{\'visible\': vis(1)}])])])\n    return _plotly_layout(\n        fig, f\'{EXPERIMENT_PATH.name} - GP surrogate, slice on \'\n             f\'{VAR_NAMES[d]}  (Q({VAR_NAMES[derived]}) = Qtot - sum)\',\n        [_axis_title(i) for i in free], height=880)\n\n\ndef _simplex_wireframe(total, name=\'Simplex boundary\'):\n    \'\'\'Edges of the feasible tetrahedron in the 3 plotted flow coordinates.\n\n    With the remaining reagent derived as `total - sum`, the feasible set is\n    the tetrahedron spanned by the origin and the three axis points at\n    `total` - drawing it gives the cloud a depth reference it otherwise\n    completely lacks.\n    \'\'\'\n    V = np.array([[0, 0, 0], [total, 0, 0], [0, total, 0], [0, 0, total]],\n                 dtype=float)\n    xs, ys, zs = [], [], []\n    for i, j in [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]:\n        xs += [V[i, 0], V[j, 0], None]\n        ys += [V[i, 1], V[j, 1], None]\n        zs += [V[i, 2], V[j, 2], None]\n    return go.Scatter3d(x=xs, y=ys, z=zs, mode=\'lines\', name=name,\n                        line=dict(color=\'rgba(0,0,0,0.5)\', width=4),\n                        hoverinfo=\'skip\')\n\n\ndef cloud_range(values, focus):\n    \'\'\'Colour range = the region the shells actually cover.\'\'\'\n    v = np.asarray(values, dtype=float)\n    v = v[np.isfinite(v)]\n    qmax = max(s[0] for s in CLOUD_SHELLS)\n    if focus == \'low\':\n        return float(v.min()), float(np.quantile(v, qmax))\n    return float(np.quantile(v, 1.0 - qmax)), float(v.max())\n\n\ndef _cloud_traces(comp, values, free, cmap, label, focus=\'low\', visible=True,\n                  cmin=None, cmax=None):\n    \'\'\'Nested point-cloud shells over the informative tail of the field.\n\n    One trace per CLOUD_SHELLS band, all sharing the same colour range so the\n    core reads as a bright dense centre inside a faint halo instead of one\n    flat blob.\n    \'\'\'\n    v = np.asarray(values, dtype=float)\n    comp = np.asarray(comp, dtype=float)\n    ok = np.isfinite(v)\n    vv = v[ok]\n    lo, hi = cloud_range(vv, focus)\n    if cmin is not None:\n        lo = float(cmin)\n    if cmax is not None:\n        hi = float(cmax)\n    names = [VAR_NAMES[i] for i in free]\n    hover = (f\'Q({names[0]})=%{{x:.2f}}<br>Q({names[1]})=%{{y:.2f}}<br>\'\n             f\'Q({names[2]})=%{{z:.2f}}<br>\'\n             f\'{label}=%{{marker.color:.4g}}<extra></extra>\')\n\n    traces, prev_q = [], 0.0\n    for k, (q, opacity, size) in enumerate(CLOUD_SHELLS):\n        if focus == \'low\':\n            hi_t = np.quantile(vv, q)\n            sel = ok & (v <= hi_t)\n            if prev_q > 0:\n                sel &= v > np.quantile(vv, prev_q)\n        else:\n            lo_t = np.quantile(vv, 1.0 - q)\n            sel = ok & (v >= lo_t)\n            if prev_q > 0:\n                sel &= v < np.quantile(vv, 1.0 - prev_q)\n        prev_q = q\n        idx = np.flatnonzero(sel)\n        if CLOUD_MAX_PTS and idx.size > CLOUD_MAX_PTS:\n            idx = RNG.choice(idx, CLOUD_MAX_PTS, replace=False)\n        pts = np.round(comp[idx][:, free], 3)\n        marker = dict(size=size, color=np.round(v[idx], 5), colorscale=cmap,\n                      cmin=lo, cmax=hi, opacity=opacity, line=dict(width=0),\n                      showscale=(k == 0))\n        if k == 0:                       # one colourbar for the whole field\n            marker[\'colorbar\'] = dict(\n                title=dict(text=label, side=\'right\', font=dict(size=19)),\n                tickfont=dict(size=15), len=0.72)\n        traces.append(go.Scatter3d(\n            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode=\'markers\',\n            marker=marker, visible=visible, hovertemplate=hover,\n            legendgroup=label, showlegend=(k == 0),\n            name=f\'{label} (best {CLOUD_SHELLS[-1][0]:.0%})\'))\n    return traces\n\n\ndef _field_buttons(groups, n_static):\n    \'\'\'Toggle buttons for fields whose traces occupy indices 0..n_field-1.\n\n    `groups` = one list of trace indices per field (a field can own several\n    traces - the cloud shells).\n    \'\'\'\n    n_field = sum(len(g) for g in groups)\n\n    def vis(k):\n        out = [False] * n_field\n        for i in groups[k]:\n            out[i] = True\n        return out + [True] * n_static\n\n    return [dict(\n        # inside the plotting area (y<=1), not up in the title margin\n        type=\'buttons\', direction=\'right\', x=0.5, xanchor=\'center\',\n        y=1.0, yanchor=\'top\', showactive=True, font=dict(size=16),\n        pad=dict(t=2, b=2, l=4, r=4), bgcolor=\'rgba(255,255,255,0.85)\',\n        buttons=[dict(label=\'posterior mean\', method=\'update\',\n                      args=[{\'visible\': vis(0)}]),\n                 dict(label=\'posterior std\', method=\'update\',\n                      args=[{\'visible\': vis(1)}])])]\n\n\ndef plotly_cloud_4pumps():\n    \'\'\'4 varying pumps: nested point-cloud shells over the 3-simplex.\'\'\'\n    free = [0, 1, 2]\n    comp = sample_simplex(CLOUD_N_4D)\n    mu, sd = gp_eval(comp)\n\n    fig = go.Figure()\n    mu_tr = _cloud_traces(comp, mu, free, \'Viridis_r\', COST_LABEL,\n                          focus=\'low\', visible=True)\n    sd_tr = _cloud_traces(comp, sd, free, \'Magma\', \'GP standard deviation\',\n                          focus=\'high\', visible=False)\n    for t in mu_tr + sd_tr:\n        fig.add_trace(t)\n    groups = [list(range(len(mu_tr))),\n              list(range(len(mu_tr), len(mu_tr) + len(sd_tr)))]\n    n_field = len(mu_tr) + len(sd_tr)\n\n    fig.add_trace(_simplex_wireframe(QTOT_VAR))\n    ghost3 = COMP_GHOST[:, free] if N_GHOST else None\n    for t in _measure_traces(COMP_REAL[:, free], ghost3):\n        fig.add_trace(t)\n\n    fig.update_layout(updatemenus=_field_buttons(groups,\n                                                 len(fig.data) - n_field))\n    return _plotly_layout(\n        fig, f\'{EXPERIMENT_PATH.name} - GP surrogate, quaternary simplex\',\n        [_axis_title(i) for i in free],\n        subtitle=f\'Q({VAR_NAMES[3]}) = Qtot - sum  |  best \'\n                 f\'{CLOUD_SHELLS[-1][0]:.0%} of the field shown\')\n\n\ndef plotly_cloud_5pumps(slice_dim=None):\n    \'\'\'5 varying pumps: point cloud over 3 reagents, slider on one of the\n    others. Each slice is its own sub-simplex, shrinking as the slider rises.\n    \'\'\'\n    d = SLICE_DIM if slice_dim is None else int(slice_dim)\n    others = [i for i in range(N_VAR) if i != d]\n    free, derived = others[:3], others[3]\n    slices = np.linspace(0, QTOT_VAR * 0.75, N_SLICES_5D)\n\n    packed = []\n    for s in slices:\n        rem = QTOT_VAR - s\n        sub = RNG.dirichlet(np.ones(4), size=CLOUD_N_5D) * rem\n        comp = np.zeros((len(sub), N_VAR))\n        comp[:, others] = sub\n        comp[:, d] = s\n        mu, sd = gp_eval(comp)\n        packed.append((comp, mu, sd, s, rem))\n\n    # one colour range across every slice, else the scale jumps as you drag\n    allmu = np.concatenate([p[1] for p in packed])\n    allsd = np.concatenate([p[2] for p in packed])\n    lo_mu, hi_mu = cloud_range(allmu, \'low\')\n    lo_sd, hi_sd = cloud_range(allsd, \'high\')\n\n    n_shell = len(CLOUD_SHELLS)\n    groups = [list(range(n_shell)), list(range(n_shell, 2 * n_shell))]\n    n_field = 2 * n_shell\n\n    frames, first = [], None\n    for comp, mu, sd, s, rem in packed:\n        tr = (_cloud_traces(comp, mu, free, \'Viridis_r\', COST_LABEL,\n                            focus=\'low\', visible=True, cmin=lo_mu, cmax=hi_mu)\n              + _cloud_traces(comp, sd, free, \'Magma\',\n                              \'GP standard deviation\', focus=\'high\',\n                              visible=False, cmin=lo_sd, cmax=hi_sd)\n              + [_simplex_wireframe(rem)])\n        frames.append(go.Frame(data=tr, traces=list(range(len(tr))),\n                               name=f\'{s:.2f}\'))\n        if first is None:\n            first = tr\n\n    fig = go.Figure(data=first, frames=frames)\n    ghost3 = COMP_GHOST[:, free] if N_GHOST else None\n    for t in _measure_traces(COMP_REAL[:, free], ghost3):\n        fig.add_trace(t)\n\n    fig.update_layout(\n        sliders=[dict(\n            active=0, x=0.08, len=0.84, y=0.02, pad=dict(t=40),\n            currentvalue=dict(prefix=f\'Q({VAR_NAMES[d]}) = \', suffix=\' \',\n                              font=dict(size=20)),\n            font=dict(size=15),\n            steps=[dict(method=\'animate\', label=f\'{s:.2f}\',\n                        args=[[f\'{s:.2f}\'],\n                              dict(mode=\'immediate\',\n                                   frame=dict(duration=0, redraw=True),\n                                   transition=dict(duration=0))])\n                   for s in slices])],\n        updatemenus=_field_buttons(groups, len(fig.data) - n_field))\n    return _plotly_layout(\n        fig, f\'{EXPERIMENT_PATH.name} - GP surrogate, slice on {VAR_NAMES[d]}\',\n        [_axis_title(i) for i in free], height=880,\n        subtitle=f\'Q({VAR_NAMES[derived]}) = Qtot - sum  |  best \'\n                 f\'{CLOUD_SHELLS[-1][0]:.0%} of the field shown\')\n\n\ndef make_interactive():\n    made = {}\n    if not HAS_PLOTLY:\n        print(\'plotly missing -> skipped\')\n        return made\n    cloud = str(INTERACTIVE_3D_MODE).lower() == \'cloud\'\n    if N_VAR == 3:\n        made[\'gp_surface_3pumps\'] = plotly_surface_3pumps()\n    elif N_VAR == 4:\n        made[\'gp_volume_4pumps\'] = (plotly_cloud_4pumps() if cloud\n                                    else plotly_volume_4pumps())\n    elif N_VAR >= 5:\n        print(f\'slider on {VAR_NAMES[SLICE_DIM]} ({N_SLICES_5D} slices, \'\n              + (f\'{CLOUD_N_5D} pts/slice, best {CLOUD_SHELLS[-1][0]:.0%} kept)\'\n                 if cloud else f\'{RES_VOLUME_5D}^3 grid)\'))\n        made[\'gp_volume_5pumps_slider\'] = (plotly_cloud_5pumps() if cloud\n                                           else plotly_volume_5pumps())\n    else:\n        print(f\'{N_VAR} varying pumps -> no interactive figure\')\n    if cloud and N_VAR >= 4:\n        print(f\'interactive 3-D mode: cloud (INTERACTIVE_3D_MODE) - set it to \'\n              f"\'volume\' for the isosurface rendering")\n    for name, fig in made.items():\n        save_html(fig, name)\n    return made\n\n\nINTERACTIVE_FIGS = make_interactive()',
    'def kernel_hparams(gp):\n    \'\'\'{hyper-parameter name: value} of a fitted sklearn kernel.\'\'\'\n    k = gp.kernel_\n    vals = np.exp(k.theta)\n    out, i = {}, 0\n    for h in k.hyperparameters:\n        n = int(h.n_elements)\n        v = vals[i:i + n]\n        i += n\n        if n == 1:\n            out[h.name] = float(v[0])\n        else:\n            for j, vv in enumerate(v):\n                out[f\'{h.name}[{j}]\'] = float(vv)\n    return out\n\n\ndef kernel_history():\n    hist = {}\n    for it in GPD[\'_available\']:\n        try:\n            d = load_gp(EXPERIMENT_PATH, it)\n        except Exception:\n            continue\n        for k, v in kernel_hparams(d[\'gaussian_process\']).items():\n            hist.setdefault(k, {})[it] = v\n    return hist\n\n\ndef _nice_hp_label(name):\n    lab = name.replace(\'k1__k1__constant_value\', \'amplitude\')\n    lab = lab.replace(\'k2__noise_level\', \'noise level\')\n    lab = lab.replace(\'k1__k2__length_scale\', \'length scale\')\n    lab = lab.replace(\'k1__length_scale\', \'length scale\')\n    lab = lab.replace(\'k1__constant_value\', \'amplitude\')\n    lab = lab.replace(\'k1__alpha\', \'alpha\')\n    return lab.replace(\'__\', \' \')\n\n\ndef fig_cost_evolution(logy=None):\n    \'\'\'Cost per evaluation, on its own. x starts at 0, init phase shaded.\'\'\'\n    m = len(COSTS_PLOT)\n    it = np.arange(m)\n    ni = min(N_INIT, m)\n    y = np.where(np.isfinite(COSTS_PLOT), COSTS_PLOT, np.nan)\n    best = np.fmin.accumulate(y)\n    pos = y[np.isfinite(y) & (y > 0)]\n    if logy is None:\n        logy = pos.size > 1 and (pos.max() / pos.min()) > 30\n\n    fig, ax = plt.subplots(figsize=(FIG_SIZE, FIG_SIZE))\n    if ni:\n        ax.axvspan(-0.5, ni - 0.5, color=\'0.88\', zorder=0, lw=0)\n        ax.annotate(f\'initial design ({ni})\', xy=((ni - 1) / 2.0, 0.985),\n                    xycoords=(\'data\', \'axes fraction\'), ha=\'center\',\n                    va=\'top\', fontsize=15, color=\'0.35\')\n\n    ax.step(it, best, where=\'post\', color=C_BEST, lw=3.0, zorder=4,\n            label=\'Best so far\')\n    m_init = (it < ni) & ~IS_FLAT[:m]\n    m_bo = (it >= ni) & ~IS_FLAT[:m]\n    if m_init.any():\n        ax.scatter(it[m_init], y[m_init], s=170, marker=\'s\', c=C_INIT,\n                   edgecolors=\'black\', linewidths=1.5, zorder=5,\n                   label=f\'Initialization ({int(m_init.sum())})\')\n    if m_bo.any():\n        ax.scatter(it[m_bo], y[m_bo], s=150, marker=\'o\', c=C_BO,\n                   edgecolors=\'black\', linewidths=1.5, zorder=5,\n                   label=f\'BO iterations ({int(m_bo.sum())})\')\n    if IS_FLAT[:m].any():\n        f = IS_FLAT[:m]\n        ax.scatter(it[f], y[f], s=180, marker=\'v\', c=\'none\', edgecolors=C_FLAT,\n                   linewidths=2.6, zorder=5,\n                   label=f\'Flat spectrum, remapped ({int(f.sum())})\')\n    ax.scatter([BEST_I], [y[BEST_I]], s=560, marker=\'*\', c=C_BEST,\n               edgecolors=\'black\', linewidths=1.5, zorder=6,\n               label=f\'Best (cost = {COSTS_ALL[BEST_I]:.4g})\')\n\n    if logy:\n        ax.set_yscale(\'log\')\n        ax.set_ylim(top=float(np.nanmax(y)) * 12)\n    else:\n        ax.set_ylim(bottom=0, top=float(np.nanmax(y)) * 1.45)\n    ax.set_xlim(-0.5, m - 0.5)\n    ax.set_xlabel(\'Evaluation\')\n    ax.set_ylabel(f\'Cost ({cost_descr()})\')\n    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))\n    ax.tick_params(direction=\'in\', top=True, right=True)\n    ax.grid(True, which=\'major\', axis=\'y\', alpha=0.25, lw=0.9)\n    style_legend(ax, loc=\'upper right\', ncol=2, fontsize=14)\n    ax.set_title(f\'{EXPERIMENT_PATH.name} - optimisation progress \'\n                 f\'({CFG.get("acq_func", "?")} acquisition)\', pad=14)\n    fig.tight_layout()\n    return fig\n\n\ndef fig_kernel_evolution():\n    \'\'\'Fitted GP hyper-parameters across the run.\'\'\'\n    hist = kernel_history()\n    if not hist:\n        return None\n    fig, ax = plt.subplots(figsize=(FIG_SIZE, FIG_SIZE))\n    colors = plt.get_cmap(\'tab10\').colors\n    for j, (name, series) in enumerate(sorted(hist.items())):\n        xs = sorted(series)\n        ax.plot(xs, [series[x] for x in xs], marker=\'o\', ms=8, lw=2.6,\n                color=colors[j % 10], label=_nice_hp_label(name))\n    ax.set_yscale(\'log\')\n    ax.set_xlabel(\'BO iteration\')\n    ax.set_ylabel(\'Fitted hyper-parameter\')\n    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))\n    ax.tick_params(direction=\'in\', top=True, right=True)\n    ax.grid(True, which=\'major\', alpha=0.25, lw=0.9)\n    style_legend(ax, loc=\'best\')\n    ax.set_title(f\'{EXPERIMENT_PATH.name} - GP hyper-parameters\', pad=14)\n    fig.tight_layout()\n    return fig\n\n\nCOST_FIG = fig_cost_evolution()\nsave_fig(COST_FIG, \'cost_evolution\')\nKERNEL_FIG = fig_kernel_evolution()\nif KERNEL_FIG is not None:\n    save_fig(KERNEL_FIG, \'kernel_evolution\')\nplt.show()',
    'SPECTRUM_STYLES = (\'full_spectrum\', \'pca\')\n\n\ndef target_is_spectrum():\n    if TARGET_IS_SPECTRUM is not None:\n        return bool(TARGET_IS_SPECTRUM)\n    style = str(CFG.get(\'target_style\', \'\')).lower()\n    return style in SPECTRUM_STYLES and TARGET_ABS is not None\n\n\ndef fig_best_spectrum():\n    if not SPEC_OK:\n        print(\'no per-evaluation spectra -> skipped\')\n        return None\n    wl = SPEC[\'wl\']\n    y = SPEC[\'spectra\'][BEST_I]\n    lmax = float(SPEC[\'lambda_max\'][BEST_I])   # stored, not recomputed - see load_spectra\n    amax = float(np.interp(lmax, wl, y))       # curve value at lmax, so the marker sits on it\n    name = SPEC[\'names\'][BEST_I]\n    is_pca = TARGET_STYLE == \'pca\'\n    pca_ready = is_pca and PCA_MODEL is not None and PCA_TARGET_SPEC is not None\n\n    fig, ax = plt.subplots(figsize=(FIG_SIZE, FIG_SIZE))\n    ax.plot(wl, y, color=C_BO, lw=3.0, zorder=5,\n            label=f\'Best measurement (#{BEST_I + POINT_LABEL_BASE}, {name})\')\n\n    if pca_ready:\n        # the actual cost basis: L2 distance in PCA score space, not a raw\n        # spectrum overlay - see pca_score()/pca_cost() in section 3b\n        ax.plot(PCA_TARGET_WL, PCA_TARGET_SPEC, color=C_TARGET, lw=2.6,\n                ls=\'--\', zorder=4, label=\'PCA target (raw, last reading)\')\n        tl = float(PCA_TARGET_WL[int(np.argmax(PCA_TARGET_SPEC))])\n        ax.axvline(tl, color=C_TARGET, lw=1.4, ls=\':\', zorder=2)\n        pca_c = pca_cost(pca_score(wl, y, PCA_MODEL),\n                         pca_score(PCA_TARGET_WL, PCA_TARGET_SPEC, PCA_MODEL))\n        ax.annotate(f\'target $\\\\lambda_{{max}}$ = {tl:.0f} nm\\n\'\n                    f\'PCA cost (recomputed) = {pca_c:.4g}\',\n                    xy=(tl, 0.0), xycoords=(\'data\', \'axes fraction\'),\n                    xytext=(10, 12), textcoords=\'offset points\',\n                    fontsize=14, color=C_TARGET, fontweight=\'bold\')\n    elif target_is_spectrum():\n        tw, ta = TARGET_WL, TARGET_ABS\n        if tw is None or len(tw) != len(ta):\n            tw = wl\n        label = (\'Target spectrum (NOT the pca cost basis - set \'\n                 \'PCA_MODEL_PATH/PCA_TARGET_PATH)\' if is_pca else \'Target spectrum\')\n        ax.plot(tw, ta, color=C_TARGET, lw=2.6, ls=\'--\', zorder=4, label=label)\n        tl = float(tw[int(np.argmax(ta))])\n        ax.axvline(tl, color=C_TARGET, lw=1.4, ls=\':\', zorder=2)\n        ax.annotate(f\'target $\\\\lambda_{{max}}$ = {tl:.0f} nm\',\n                    xy=(tl, 0.0), xycoords=(\'data\', \'axes fraction\'),\n                    xytext=(10, 12), textcoords=\'offset points\',\n                    fontsize=15, color=C_TARGET, fontweight=\'bold\')\n    else:\n        # tg_abs is a config artefact, not actually used by the BO -> not plotted\n        tw = float(CFG.get(\'tg_wl\', np.nan))\n        if np.isfinite(tw):\n            ax.axvline(tw, color=C_TARGET, lw=3.0, ls=\'--\', zorder=4,\n                       label=f\'Target $\\\\lambda_{{max}}$ = {tw:.1f} nm\')\n\n    ax.axvline(lmax, color=C_BEST, lw=2.2, ls=\'-.\', zorder=3,\n               label=f\'Best $\\\\lambda_{{max}}$ = {lmax:.1f} nm\')\n    ax.plot([lmax], [amax], marker=\'*\', ms=26, color=C_BEST,\n            markeredgecolor=\'black\', markeredgewidth=1.4, zorder=7, ls=\'none\')\n    ax.annotate(f\'$\\\\lambda_{{max}}$ = {lmax:.1f} nm\\nA = {amax:.3f}\',\n                xy=(lmax, amax), xytext=(-16, -58), ha=\'right\',\n                textcoords=\'offset points\', fontsize=16, fontweight=\'bold\',\n                color=C_BEST,\n                path_effects=[pe.withStroke(linewidth=3.0, foreground=\'white\')])\n\n    if SPECTRUM_XLIM:\n        ax.set_xlim(*SPECTRUM_XLIM)\n    ax.set_ylim(bottom=min(0.0, float(np.nanmin(y))))\n    ax.set_xlabel(\'Wavelength (nm)\')\n    ax.set_ylabel(\'Extinction\')\n    ax.tick_params(direction=\'in\', top=True, right=True)\n    ax.grid(True, alpha=0.22, lw=0.9)\n    style_legend(ax, loc=\'best\', fontsize=14)\n    ax.set_title(f\'{EXPERIMENT_PATH.name} - best spectrum \'\n                 f\'(cost = {COSTS_ALL[BEST_I]:.4g}, \'\n                 f\'{CFG.get("target_style", "?")} target)\', pad=14)\n    fig.tight_layout()\n    return fig\n\n\nSPECTRUM_FIG = fig_best_spectrum()\nif SPECTRUM_FIG is not None:\n    save_fig(SPECTRUM_FIG, \'best_spectrum\')\nplt.show()',
    "def fig_best_spectrum_normalized():\n    is_pca = TARGET_STYLE == 'pca'\n    pca_ready = is_pca and PCA_MODEL is not None and PCA_TARGET_SPEC is not None\n    if not (SPEC_OK and pca_ready):\n        print('pca target not configured (PCA_MODEL_PATH/PCA_TARGET_PATH) -> skipped')\n        return None\n\n    wl = SPEC['wl']\n    y = SPEC['spectra'][BEST_I]\n    y_n = y / y.max()\n    tw, ta = PCA_TARGET_WL, PCA_TARGET_SPEC\n    ta_n = ta / ta.max()\n    lmax = float(SPEC['lambda_max'][BEST_I])\n    amax_n = float(np.interp(lmax, wl, y_n))\n    tl = float(tw[int(np.argmax(ta))])\n    pca_c = pca_cost(pca_score(wl, y, PCA_MODEL),\n                     pca_score(tw, ta, PCA_MODEL))\n\n    fig, ax = plt.subplots(figsize=(FIG_SIZE, FIG_SIZE))\n    ax.plot(wl, y_n, color=C_BO, lw=3.0, zorder=5,\n            label=f'Best measurement (#{BEST_I + POINT_LABEL_BASE}), normalized')\n    ax.plot(tw, ta_n, color=C_TARGET, lw=2.6, ls='--', zorder=4,\n            label='PCA target, normalized')\n    ax.axvline(tl, color=C_TARGET, lw=1.4, ls=':', zorder=2)\n    ax.axvline(lmax, color=C_BEST, lw=2.2, ls='-.', zorder=3,\n               label=f'Best $\\\\lambda_{{max}}$ = {lmax:.1f} nm')\n    ax.plot([lmax], [amax_n], marker='*', ms=26, color=C_BEST,\n            markeredgecolor='black', markeredgewidth=1.4, zorder=7, ls='none')\n    ax.annotate(f'target $\\\\lambda_{{max}}$ = {tl:.0f} nm\\n'\n                f'PCA cost (recomputed) = {pca_c:.4g}',\n                xy=(tl, 0.0), xycoords=('data', 'axes fraction'),\n                xytext=(10, 12), textcoords='offset points',\n                fontsize=14, color=C_TARGET, fontweight='bold')\n\n    if SPECTRUM_XLIM:\n        ax.set_xlim(*SPECTRUM_XLIM)\n    ax.set_ylim(0.0, 1.05)\n    ax.set_xlabel('Wavelength (nm)')\n    ax.set_ylabel('Extinction / max(Extinction)')\n    ax.tick_params(direction='in', top=True, right=True)\n    ax.grid(True, alpha=0.22, lw=0.9)\n    style_legend(ax, loc='upper right', fontsize=14)\n    ax.set_title(f'{EXPERIMENT_PATH.name} - best spectrum vs. target '\n                 f'(PCA-normalized, cost = {COSTS_ALL[BEST_I]:.4g})', pad=14)\n    fig.tight_layout()\n    return fig\n\n\nSPECTRUM_NORM_FIG = fig_best_spectrum_normalized()\nif SPECTRUM_NORM_FIG is not None:\n    save_fig(SPECTRUM_NORM_FIG, 'best_spectrum_normalized')\nplt.show()",
    "print(f'Everything below is in  {OUTPUT_DIR}\\n')\nfor p in sorted(OUTPUT_DIR.iterdir()):\n    print(f'  {p.name:<42s} {p.stat().st_size / 1024:8.0f} kB')",
]


### `names`

In [ ]:
"""Names shown on the figures: the run of a path, and "S31-M01 sample_1" for a sample."""
import re


def new_name(old):
    name = str(old).replace("\\", "/").strip("/").split("/")[-1]
    return re.sub(r"_((?:sample|init|iter)_\d+)$", r" \1", name)


label = new_name


### `bo_publi`

In [ ]:
"""Figures S16–S22 — individual BO runs, drawn by the ORIGINAL generic notebook paperAnalysis/BO/bo_figure_publi.ipynb
(code cells vendored verbatim in bo_publi_cells.py).

Each run is executed in a fresh namespace (papermill-like); only the configuration lines below are replaced:
  EXPERIMENT_PATH -> raw/<run> (read-only), OUTPUT_DIR -> figures/bo_publi/<run> (never inside raw/),
  PCA_MODEL_PATH / PCA_TARGET_PATH for the PCA-target runs (frozen bases delivered in data/inputs/), SAVE_HTML -> False
  (the interactive HTML exports are not paper figures), STOP_AT_EVAL for the one run whose figure stops early (S21).

Panels per SI figure (file names of the notebook): S16 gp_mean_2pumps (1-D, mean ± 2σ) | S17–S19 gp_mean_ternary_3pumps |
S20–S22 gp_mean_corner; all: cost_evolution + best_spectrum (best_spectrum_normalized for the PCA runs).
"""
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt

from toofast_analysis.bo_publi_cells import CELLS

# run -> (folder under raw/, PCA model, PCA target under raw/). PCA settings = the ones that reproduce costs.csv
# (summary.txt "PCA TARGET" block of the original figures_publi/, except S19-M04, see below).
RUNS = {
    "S16": dict(run="S04_2/S04-M03"),
    "S17": dict(run="S04_2/S04-M02"),
    # S19-M04: the last original summary.txt (2026-09-30) used 2FAST_ML/PCA/target_spectrum.csv and did NOT reproduce
    # costs.csv (ratio 1.14 ± 3.9); the run's target is S19-M02 sample_2 (reproduction ratio 0.9996 ± 0.0007).
    "S18": dict(run="S19/S19-M04", pca="pca_model_norm6.npz",
                target="S19/S19-M02/measurement_spectra_sample_2_copy.csv"),
    "S19": dict(run="S22_2/S22-M03", pca="pca_model_5.npz", target="S23/S23-M05/measurement_spectra_iter_1.csv"),
    "S20": dict(run="S08/S08-M02"),
    # S25-M03: 20 evaluations in raw/, the SI figure stops at the best (#15, 0-based) = 16 evaluations, GP iter 10
    # (original summary.txt; generator comment "S25-M03's best is #15 -> STOP_AT_EVAL = 15").
    "S21": dict(run="S25-M03", pca="pca_model_5.npz", target="S23/S23-M05/measurement_spectra_iter_1.csv",
                stop_at_eval=15),
    "S22": dict(run="S28-M02", pca="pca_model_5.npz", target="S23/S23-M05/measurement_spectra_iter_1.csv"),
}

_ORIG = {
    "path": "EXPERIMENT_PATH = Path(r'C:\\Users\\user\\paperAnalysis\\BO\\S28-M01')\n",
    "pca": "PCA_MODEL_PATH = None     # e.g. r'C:\\Users\\user\\2FAST_ML\\PCA\\pca_model_5.npz'\n",
    "target": "PCA_TARGET_PATH = None    # e.g. r'C:\\...\\S22-M01\\measurement_spectra_iter_1.csv'\n",
    "out": "OUTPUT_DIR = EXPERIMENT_PATH / OUTPUT_SUBDIR\n",
    "html": "SAVE_HTML = True\n",
    "stop": "STOP_AT_EVAL = None     # cut the run at this evaluation number (as printed on\n",
}


def config_cell(spec, out_dir, raw=None):
    raw = RAW if raw is None else raw
    src = CELLS[1]
    new = {
        "path": f"EXPERIMENT_PATH = Path(r'{raw / spec['run']}')   # [toofast_analysis] raw/, read-only\n",
        "pca": f"PCA_MODEL_PATH = {str(INPUTS / spec['pca']) if 'pca' in spec else None!r}   # [toofast_analysis]\n",
        "target": f"PCA_TARGET_PATH = {str(raw / spec['target']) if 'target' in spec else None!r}   # [toofast_analysis]\n",
        "out": f"OUTPUT_DIR = Path(r'{out_dir}')   # [toofast_analysis] never inside raw/\n",
        "html": "SAVE_HTML = False   # [toofast_analysis] interactive exports are not paper figures\n",
        "stop": f"STOP_AT_EVAL = {spec.get('stop_at_eval')!r}   # [toofast_analysis] cut the run at this evaluation number (as printed on\n",
    }
    for k, old in _ORIG.items():
        if src.count(old) != 1:
            raise RuntimeError(f"configuration line not found exactly once: {old!r}")
        src = src.replace(old, new[k])
    return src


def summarise(ns):
    """Numbers quoted in the SI text, from a run's namespace (evaluation indices 0-based, as printed on the figures)."""
    import numpy as np
    bi = int(ns["BEST_I"])
    out = dict(n_eval=len(ns["COSTS_ALL"]), n_init=int(ns["N_INIT"]), n_virtual=int(ns["N_GHOST"]),
               n_flat=int(np.sum(ns["IS_FLAT"])), best_eval=bi, best_cost=float(ns["COSTS_ALL"][bi]),
               best_flows=[float(v) for v in ns["FLOWS_ALL"][bi]], reagents=[str(r) for r in ns["REAGENTS"]],
               kernel=str(ns["GP"].kernel_), gp_file=str(ns.get("GP_FILE", "")), target_style=ns["TARGET_STYLE"])
    if ns.get("SPEC_OK"):
        spec = ns["SPEC"]
        out["best_lambda_max"] = float(spec["lambda_max"][bi])
        if ns.get("PCA_TARGET_SPEC") is not None:
            wl, y = spec["wl"], spec["spectra"][bi]
            tw, ta = ns["PCA_TARGET_WL"], ns["PCA_TARGET_SPEC"]
            t = np.interp(wl, tw, ta)
            out["target_lambda_max"] = float(tw[int(np.argmax(ta))])
            out["raw_cosine_best_vs_target"] = float(y @ t / (np.linalg.norm(y) * np.linalg.norm(t)))
            out["max_ratio_best_over_target"] = float(np.max(y) / np.max(ta))
    tg = ns["CFG"].get("tg_wl")
    if ns["TARGET_STYLE"] == "lambda_max" and tg is not None:
        out["target_lambda_max"] = float(tg)
    return out


def main(figs=None, out_root=None):
    """Run every SI figure and write data/processed/bo_publi_summary.json."""
    import json
    pass
    res = {}
    for fig in figs or RUNS:
        print(f"===== {fig} {RUNS[fig]['run']}", flush=True)
        res[fig] = dict(run=RUNS[fig]["run"], **summarise(run(fig, out_root)))
    PROCESSED.mkdir(parents=True, exist_ok=True)
    (PROCESSED / "bo_publi_summary.json").write_text(json.dumps(res, indent=1), encoding="utf-8")
    return res


def run(fig, out_root=None, raw=None):
    """Execute the notebook for one SI figure; returns its namespace (SUMMARY_LINES, COSTS_ALL, BEST_I, ...)."""
    spec = RUNS[fig]
    out_dir = Path(out_root or FIGURES / "bo_publi") / new_name(spec["run"])     # e.g. figures/bo_publi/S04-M03
    out_dir.mkdir(parents=True, exist_ok=True)
    matplotlib.use("Agg")
    ns = {"__name__": "__bo_publi__", "display": lambda *a, **k: None}
    show = plt.show
    plt.show = lambda *a, **k: None
    try:
        with matplotlib.rc_context():                    # cell 0 updates rcParams globally: keep it local to the run
            for i, cell in enumerate(CELLS):
                src = config_cell(spec, out_dir, raw) if i == 1 else cell
                exec(compile(src, f"<bo_figure_publi cell {i}>", "exec"), ns)
                plt.close("all")
    finally:
        plt.show = show
    return ns


## Figure

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

res = main()
for fig, s in res.items():
    print(f"{fig} {label(s['run'])}: {s['n_eval']} evals ({s['n_init']} init), {s['n_virtual']} virtual, {s['n_flat']} flat | "
          f"best #{s['best_eval']} cost {s['best_cost']:.4g}, lambda_max {s.get('best_lambda_max', float('nan')):.1f} nm")


In [ ]:
PANELS = {'S16': 'gp_mean_2pumps', 'S17': 'gp_mean_ternary_3pumps', 'S18': 'gp_mean_ternary_3pumps',
          'S19': 'gp_mean_ternary_3pumps', 'S20': 'gp_mean_corner', 'S21': 'gp_mean_corner', 'S22': 'gp_mean_corner'}
for fig, spec in RUNS.items():
    d = FIGURES / 'bo_publi' / new_name(spec['run'])
    spectrum = 'best_spectrum_normalized' if fig == 'S18' else 'best_spectrum'
    for name in (PANELS[fig], 'cost_evolution', spectrum):
        print(fig, name)
        display(Image(str(d / f'{name}.png'), width=520))
